<a href="https://colab.research.google.com/github/Amber-0117/Test0207/blob/main/2D_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install tensorflow

In [ ]:
import os

print(os.getcwd())
print(os.listdir('/content'))

In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

tf.random.set_seed(1)

#Data Load
import os

# 指定四季 CSV
season_files = {
    'autumn': '/content/clearautumn.csv',
    'spring': '/content/clearspring.csv',
    'summer': '/content/clearsummer.csv',
    'winter': '/content/clearwinter.csv'
}

output_dir = '/content/sample_data/season_split'
os.makedirs(output_dir, exist_ok=True)

season_data = {}

for season, file_path in season_files.items():
    df = pd.read_csv(file_path)

    # 時間資料不打亂，前 80% train，後 20% test
    split_index = int(len(df) * 0.8)

    train_raw = df.iloc[:split_index].copy()
    test_raw = df.iloc[split_index:].copy()

    # 預測目標：下一小時的 windspeed_120
    # 先切分再各自 shift，避免 train 最後一筆偷看到 test 第一筆
    train_raw['target_windspeed_1h'] = train_raw['windspeed_120'].shift(-1)
    test_raw['target_windspeed_1h'] = test_raw['windspeed_120'].shift(-1)

    # 最後一筆沒有下一小時資料，所以刪掉
    df_train = train_raw.dropna(subset=['target_windspeed_1h'])
    df_test = test_raw.dropna(subset=['target_windspeed_1h'])

    # 另存成 CSV
    train_path = f'{output_dir}/{season}_train_80.csv'
    test_path = f'{output_dir}/{season}_test_20.csv'

    df_train.to_csv(train_path, index=False)
    df_test.to_csv(test_path, index=False)

    # 模型輸入：拿掉時間與答案欄位
    train_data_input = df_train.drop(columns=['valid_time', 'target_windspeed_1h'])
    train_data_output = df_train['target_windspeed_1h']

    test_data_input = df_test.drop(columns=['valid_time', 'target_windspeed_1h'])
    test_data_output = df_test['target_windspeed_1h']

    season_data[season] = {
        'df_train': df_train,
        'df_test': df_test,
        'train_data_input': train_data_input,
        'train_data_output': train_data_output,
        'test_data_input': test_data_input,
        'test_data_output': test_data_output
    }

    print(f'===== {season} =====')
    print('train input:', train_data_input.shape)
    print('train output:', train_data_output.shape)
    print('test input:', test_data_input.shape)
    print('test output:', test_data_output.shape)
    print('saved:', train_path)
    print('saved:', test_path)

In [ ]:
# Hyper Parameter
seqLength = 6
filters = 100
kernel_size = (2, 2)
kernel_size2 = (2, 2)
pooling_size = (2, 2)
stride = (1, 1)
epochs = 100
batch_size = 32

target_col = 'target_windspeed_1h'


def sliding_window(X, y, seq_length):
    dataX = []
    dataY = []

    for i in range(0, len(X) - seq_length + 1):
        dataX.append(X[i:i + seq_length])
        dataY.append(y[i + seq_length - 1])

    return np.array(dataX), np.array(dataY).reshape(-1, 1)


def build_2d_cnn(seq_length, feature_count):
    model = Sequential()
    model.add(Conv2D(
        filters=filters,
        kernel_size=kernel_size,
        strides=stride,
        activation='relu',
        input_shape=(seq_length, feature_count, 1)
    ))
    model.add(Conv2D(
        filters=filters,
        kernel_size=kernel_size2,
        strides=stride,
        activation='relu'
    ))
    model.add(MaxPooling2D(pool_size=pooling_size))
    model.add(Flatten())
    model.add(Dense(48, activation='relu'))
    model.add(Dense(1))

    model.compile(optimizer='adam', loss='mse')
    return model


results = []
predictions = {}

plt.figure(figsize=(16, 10))

for idx, season in enumerate(['spring', 'summer', 'autumn', 'winter'], start=1):
    df_train = season_data[season]['df_train']
    df_test = season_data[season]['df_test']

    X_train = df_train.drop(columns=['valid_time', target_col])
    y_train = df_train[target_col]

    X_test = df_test.drop(columns=['valid_time', target_col])
    y_test = df_test[target_col]

    # 全部轉數字
    X_train = X_train.apply(pd.to_numeric, errors='coerce')
    X_test = X_test.apply(pd.to_numeric, errors='coerce')
    y_train = pd.to_numeric(y_train, errors='coerce')
    y_test = pd.to_numeric(y_test, errors='coerce')

    # 補 NaN
    X_train = X_train.fillna(X_train.median()).fillna(0)
    X_test = X_test.fillna(X_train.median()).fillna(0)
    y_train = y_train.fillna(y_train.median())
    y_test = y_test.fillna(y_train.median())

    # MinMax normalization，只用 train 的 min/max
    train_min = X_train.min()
    train_max = X_train.max()
    denom = train_max - train_min
    denom[denom == 0] = 1

    trainSet = ((X_train - train_min) / denom).values
    testSet = ((X_test - train_min) / denom).values

    trainLabel = y_train.values
    testLabel = y_test.values

    trainX, trainY = sliding_window(trainSet, trainLabel, seqLength)
    testX, testY = sliding_window(testSet, testLabel, seqLength)

    trainX = trainX.reshape(trainX.shape[0], trainX.shape[1], trainX.shape[2], 1)
    testX = testX.reshape(testX.shape[0], testX.shape[1], testX.shape[2], 1)

    model = build_2d_cnn(seqLength, trainX.shape[2])

    print(f'===== Training {season} model =====')
    model.fit(
        trainX,
        trainY,
        epochs=epochs,
        batch_size=batch_size,
        verbose=0
    )

    yhat = model.predict(testX, verbose=0).reshape(-1)
    real = testY.reshape(-1)

    mae = mean_absolute_error(real, yhat)
    rmse = np.sqrt(mean_squared_error(real, yhat))
    r2 = r2_score(real, yhat)

    results.append({
        'season': season,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    })

    predictions[season] = {
        'real': real,
        'estimated': yhat,
        'model': model
    }

    plt.subplot(2, 2, idx)
    plt.plot(yhat, label='Estimated by 2D-CNN')
    plt.plot(real, label='Real')
    plt.title(f'{season.capitalize()} One-hour Ahead Wind Speed')
    plt.xlabel('Test sample index')
    plt.ylabel('Wind speed')
    plt.legend()

plt.tight_layout()
plt.show()

results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
# Leakage / Overfitting Check

print("===== Leakage / Overfitting Check =====")

check_results = []

for season in ['spring', 'summer', 'autumn', 'winter']:
    print(f"\n===== {season.upper()} =====")

    df_train = season_data[season]['df_train'].copy()
    df_test = season_data[season]['df_test'].copy()

    train_input_cols = season_data[season]['train_data_input'].columns.tolist()
    test_input_cols = season_data[season]['test_data_input'].columns.tolist()

    # 1. 檢查答案欄位是否混進 X
    forbidden_cols = ['target_windspeed_1h', 'valid_time']
    leaked_cols = [col for col in forbidden_cols if col in train_input_cols]

    if leaked_cols:
        print("❌ X contains leaked columns:", leaked_cols)
    else:
        print("✅ X does not contain target/time columns")

    # 2. 檢查 train/test 時間是否重疊
    train_times = set(df_train['valid_time'])
    test_times = set(df_test['valid_time'])
    overlap_times = train_times.intersection(test_times)

    if overlap_times:
        print("❌ Train/Test time overlap:", len(overlap_times))
    else:
        print("✅ No train/test time overlap")

    # 3. 檢查 train 最後時間、test 第一時間
    print("Train time range:", df_train['valid_time'].iloc[0], "→", df_train['valid_time'].iloc[-1])
    print("Test time range :", df_test['valid_time'].iloc[0], "→", df_test['valid_time'].iloc[-1])

    # 4. 檢查欄位是否一致
    if train_input_cols == test_input_cols:
        print("✅ Train/Test input columns match")
    else:
        print("❌ Train/Test input columns do not match")

    # 5. 檢查 NaN
    train_nan = season_data[season]['train_data_input'].isna().sum().sum()
    test_nan = season_data[season]['test_data_input'].isna().sum().sum()

    if train_nan == 0 and test_nan == 0:
        print("✅ No NaN in input data")
    else:
        print("⚠️ NaN found - train:", train_nan, "test:", test_nan)

    check_results.append({
        'season': season,
        'leaked_cols': leaked_cols,
        'time_overlap_count': len(overlap_times),
        'train_nan': train_nan,
        'test_nan': test_nan
    })

check_df = pd.DataFrame(check_results)
print("\n===== Summary =====")
print(check_df)

In [ ]:
# 1D-CNN

In [ ]:
# =========================
# 1D-CNN Four Seasons Model
# =========================

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv1D, MaxPooling1D
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Hyper Parameter
seqLength = 6
filters = 100
kernel_size = 2
kernel_size2 = 2
pooling_size = 2
stride = 1
epochs = 100
batch_size = 32

target_col = 'target_windspeed_1h'


def sliding_window_1d(X, y, seq_length):
    dataX = []
    dataY = []

    for i in range(0, len(X) - seq_length + 1):
        dataX.append(X[i:i + seq_length])
        dataY.append(y[i + seq_length - 1])

    return np.array(dataX), np.array(dataY).reshape(-1, 1)


def build_1d_cnn(seq_length, feature_count):
    model = Sequential()
    model.add(Conv1D(
        filters=filters,
        kernel_size=kernel_size,
        strides=stride,
        activation='relu',
        input_shape=(seq_length, feature_count)
    ))
    model.add(Conv1D(
        filters=filters,
        kernel_size=kernel_size2,
        strides=stride,
        activation='relu'
    ))
    model.add(MaxPooling1D(pool_size=pooling_size))
    model.add(Flatten())
    model.add(Dense(48, activation='relu'))
    model.add(Dense(1))

    model.compile(optimizer='adam', loss='mse')
    return model


results_1d = []
predictions_1d = {}

plt.figure(figsize=(16, 10))

for idx, season in enumerate(['spring', 'summer', 'autumn', 'winter'], start=1):
    tf.keras.backend.clear_session()

    df_train = season_data[season]['df_train']
    df_test = season_data[season]['df_test']

    # X = 輸入資料，不包含時間與答案
    X_train = df_train.drop(columns=['valid_time', target_col])
    y_train = df_train[target_col]

    X_test = df_test.drop(columns=['valid_time', target_col])
    y_test = df_test[target_col]

    # 轉成數字
    X_train = X_train.apply(pd.to_numeric, errors='coerce')
    X_test = X_test.apply(pd.to_numeric, errors='coerce')
    y_train = pd.to_numeric(y_train, errors='coerce')
    y_test = pd.to_numeric(y_test, errors='coerce')

    # 補 NaN
    X_train = X_train.fillna(X_train.median()).fillna(0)
    X_test = X_test.fillna(X_train.median()).fillna(0)
    y_train = y_train.fillna(y_train.median())
    y_test = y_test.fillna(y_train.median())

    # MinMax normalization，只用 train 的 min/max
    train_min = X_train.min()
    train_max = X_train.max()
    denom = train_max - train_min
    denom[denom == 0] = 1

    trainSet = ((X_train - train_min) / denom).values
    testSet = ((X_test - train_min) / denom).values

    trainLabel = y_train.values
    testLabel = y_test.values

    # 1D-CNN windowing
    # trainX shape = (samples, seqLength, features)
    trainX, trainY = sliding_window_1d(trainSet, trainLabel, seqLength)
    testX, testY = sliding_window_1d(testSet, testLabel, seqLength)

    feature_count = trainX.shape[2]

    model = build_1d_cnn(seqLength, feature_count)

    print(f'===== Training {season} 1D-CNN model =====')
    model.fit(
        trainX,
        trainY,
        epochs=epochs,
        batch_size=batch_size,
        verbose=0
    )

    test_mse = model.evaluate(testX, testY, batch_size=batch_size, verbose=0)

    yhat = model.predict(testX, verbose=0).reshape(-1)
    real = testY.reshape(-1)

    mae = mean_absolute_error(real, yhat)
    rmse = np.sqrt(mean_squared_error(real, yhat))
    r2 = r2_score(real, yhat)

    results_1d.append({
        'season': season,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
        'Test_MSE': test_mse
    })

    predictions_1d[season] = {
        'real': real,
        'estimated': yhat,
        'model': model
    }

    plt.subplot(2, 2, idx)
    plt.plot(yhat, label='Estimated by 1D-CNN')
    plt.plot(real, label='Real')
    plt.title(f'{season.capitalize()} One-hour Ahead Wind Speed - 1D-CNN')
    plt.xlabel('Test sample index')
    plt.ylabel('Wind speed')
    plt.legend()

plt.tight_layout()
plt.show()

results_1d_df = pd.DataFrame(results_1d)
print(results_1d_df)

In [ ]:
pip install optuna

In [ ]:
# ============================================================
# 1D-CNN + Optuna Hyperparameter Optimization
# Summer Model
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import optuna

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout
)
from tensorflow.keras.optimizers import Adam, RMSprop, Nadam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# 1. 基本設定
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

season = 'summer'

target_col = 'target_windspeed_1h'

# Optuna trial 數量
# 不考慮時間成本的話，可以之後提高到 1000、3000、5000...
N_TRIALS = 500

# 每個 trial 最多 epoch
MAX_EPOCHS = 150

# EarlyStopping
PATIENCE = 15

# validation 比例
VALIDATION_RATIO = 0.2


# ============================================================
# 2. Sliding Window
# ============================================================

def sliding_window_1d(X, y, seq_length):
    dataX = []
    dataY = []

    for i in range(len(X) - seq_length + 1):
        dataX.append(
            X[i:i + seq_length]
        )

        dataY.append(
            y[i + seq_length - 1]
        )

    return (
        np.array(dataX),
        np.array(dataY).reshape(-1, 1)
    )


# ============================================================
# 3. 取得 Summer 資料
# ============================================================

df_train = season_data[season]['df_train'].copy()
df_test = season_data[season]['df_test'].copy()


print("===== Original Train Columns =====")
print(df_train.columns.tolist())


# ============================================================
# 4. 建立 X / y
# ============================================================

X_train_df = df_train.drop(
    columns=['valid_time', target_col]
)

y_train_series = df_train[target_col]

X_test_df = df_test.drop(
    columns=['valid_time', target_col]
)

y_test_series = df_test[target_col]


# ============================================================
# 5. 全部轉 numeric
# ============================================================

X_train_df = X_train_df.apply(
    pd.to_numeric,
    errors='coerce'
)

X_test_df = X_test_df.apply(
    pd.to_numeric,
    errors='coerce'
)

y_train_series = pd.to_numeric(
    y_train_series,
    errors='coerce'
)

y_test_series = pd.to_numeric(
    y_test_series,
    errors='coerce'
)


# ============================================================
# 6. NaN 處理
# ============================================================

train_median = X_train_df.median()

X_train_df = (
    X_train_df
    .fillna(train_median)
    .fillna(0)
)

X_test_df = (
    X_test_df
    .fillna(train_median)
    .fillna(0)
)

y_train_series = y_train_series.fillna(
    y_train_series.median()
)

y_test_series = y_test_series.fillna(
    y_train_series.median()
)


# ============================================================
# 7. 檢查 constant features
# ============================================================

feature_unique_count = X_train_df.nunique()

constant_features = feature_unique_count[
    feature_unique_count <= 1
].index.tolist()

print("\n===== Constant Features =====")

if constant_features:
    print(constant_features)
else:
    print("None")


# 如果你希望直接刪除完全沒有變化的 feature，可以打開下面兩行
# X_train_df = X_train_df.drop(columns=constant_features)
# X_test_df = X_test_df.drop(columns=constant_features)


print("\n===== CNN Input Features =====")
print(X_train_df.columns.tolist())

print("\nFeature count:")
print(X_train_df.shape[1])


# ============================================================
# 8. Min-Max Normalization
#    只使用 Train 的 min / max
# ============================================================

train_min = X_train_df.min()

train_max = X_train_df.max()

denom = train_max - train_min

denom[denom == 0] = 1


trainSet = (
    (X_train_df - train_min) / denom
).values.astype(np.float32)

testSet = (
    (X_test_df - train_min) / denom
).values.astype(np.float32)


trainLabel = y_train_series.values.astype(
    np.float32
)

testLabel = y_test_series.values.astype(
    np.float32
)


# ============================================================
# 9. Optimizer
# ============================================================

def get_optimizer(
    optimizer_name,
    learning_rate
):

    if optimizer_name == 'Adam':

        return Adam(
            learning_rate=learning_rate
        )

    elif optimizer_name == 'RMSprop':

        return RMSprop(
            learning_rate=learning_rate
        )

    elif optimizer_name == 'Nadam':

        return Nadam(
            learning_rate=learning_rate
        )

    else:

        raise ValueError(
            f"Unknown optimizer: {optimizer_name}"
        )


# ============================================================
# 10. 計算 CNN 時間維度是否合法
#     padding='valid'
# ============================================================

def conv_output_length(
    input_length,
    kernel_size,
    stride
):

    return (
        (input_length - kernel_size)
        // stride
    ) + 1


def pool_output_length(
    input_length,
    pool_size
):

    return (
        (input_length - pool_size)
        // pool_size
    ) + 1


# ============================================================
# 11. Optuna Objective
# ============================================================

def objective(trial):

    tf.keras.backend.clear_session()

    # --------------------------------------------------------
    # A. Sequence Length
    # 1 ~ 72
    # --------------------------------------------------------

    seq_length = trial.suggest_int(
        'seqLength',
        1,
        72
    )


    # --------------------------------------------------------
    # B. CNN 層數
    # --------------------------------------------------------

    n_conv_layers = trial.suggest_int(
        'n_conv_layers',
        1,
        4
    )


    # --------------------------------------------------------
    # C. Pooling
    # --------------------------------------------------------

    use_pooling = trial.suggest_categorical(
        'use_pooling',
        [True, False]
    )

    pooling_size = None

    if use_pooling:

        pooling_size = trial.suggest_int(
            'pooling_size',
            1,
            4
        )


    # --------------------------------------------------------
    # D. Dense 層數
    # --------------------------------------------------------

    n_dense_layers = trial.suggest_int(
        'n_dense_layers',
        1,
        3
    )


    # --------------------------------------------------------
    # E. Activation
    # --------------------------------------------------------

    activation = trial.suggest_categorical(
        'activation',
        [
            'relu',
            'elu',
            'selu',
            'tanh'
        ]
    )


    # --------------------------------------------------------
    # F. Dropout
    # --------------------------------------------------------

    dropout_rate = trial.suggest_float(
        'dropout_rate',
        0.0,
        0.5,
        step=0.05
    )


    # --------------------------------------------------------
    # G. Optimizer
    # --------------------------------------------------------

    optimizer_name = trial.suggest_categorical(
        'optimizer',
        [
            'Adam',
            'RMSprop',
            'Nadam'
        ]
    )


    # --------------------------------------------------------
    # H. Learning Rate
    # --------------------------------------------------------

    learning_rate = trial.suggest_float(
        'learning_rate',
        1e-5,
        1e-2,
        log=True
    )


    # --------------------------------------------------------
    # I. Batch Size
    # --------------------------------------------------------

    batch_size = trial.suggest_int(
        'batch_size',
        8,
        256,
        step=8
    )


    # ========================================================
    # Sliding window
    # ========================================================

    if len(trainSet) <= seq_length:

        raise optuna.TrialPruned()


    fullX, fullY = sliding_window_1d(
        trainSet,
        trainLabel,
        seq_length
    )


    if len(fullX) < 20:

        raise optuna.TrialPruned()


    # ========================================================
    # Time-series Train / Validation split
    # 不 shuffle
    # ========================================================

    split_index = int(
        len(fullX)
        * (1 - VALIDATION_RATIO)
    )


    opt_trainX = fullX[:split_index]
    opt_trainY = fullY[:split_index]

    opt_valX = fullX[split_index:]
    opt_valY = fullY[split_index:]


    if (
        len(opt_trainX) == 0
        or
        len(opt_valX) == 0
    ):

        raise optuna.TrialPruned()


    # ========================================================
    # 建立模型
    # ========================================================

    model = Sequential()

    model.add(
        Input(
            shape=(
                seq_length,
                trainSet.shape[1]
            )
        )
    )


    current_length = seq_length


    # ========================================================
    # Dynamic Conv1D Layers
    # ========================================================

    for layer_idx in range(
        n_conv_layers
    ):

        filters = trial.suggest_int(
            f'filters_{layer_idx + 1}',
            8,
            256,
            step=8
        )


        kernel_size = trial.suggest_int(
            f'kernel_size_{layer_idx + 1}',
            1,
            12
        )


        stride = trial.suggest_int(
            f'stride_{layer_idx + 1}',
            1,
            3
        )


        # 計算是否還能做 convolution
        new_length = conv_output_length(
            current_length,
            kernel_size,
            stride
        )


        if new_length < 1:

            raise optuna.TrialPruned()


        model.add(
            Conv1D(
                filters=filters,
                kernel_size=kernel_size,
                strides=stride,
                activation=activation,
                padding='valid'
            )
        )


        current_length = new_length


    # ========================================================
    # Pooling
    # ========================================================

    if use_pooling:

        new_length = pool_output_length(
            current_length,
            pooling_size
        )


        if new_length < 1:

            raise optuna.TrialPruned()


        model.add(
            MaxPooling1D(
                pool_size=pooling_size
            )
        )


        current_length = new_length


    # ========================================================
    # Flatten
    # ========================================================

    model.add(
        Flatten()
    )


    # ========================================================
    # Dynamic Dense Layers
    # ========================================================

    for dense_idx in range(
        n_dense_layers
    ):

        dense_units = trial.suggest_int(
            f'dense_units_{dense_idx + 1}',
            8,
            512,
            step=8
        )


        model.add(
            Dense(
                dense_units,
                activation=activation
            )
        )


        if dropout_rate > 0:

            model.add(
                Dropout(
                    dropout_rate
                )
            )


    # Output layer
    model.add(
        Dense(1)
    )


    # ========================================================
    # Compile
    # ========================================================

    optimizer = get_optimizer(
        optimizer_name,
        learning_rate
    )


    model.compile(
        optimizer=optimizer,
        loss='mse'
    )


    # ========================================================
    # Early stopping
    # ========================================================

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=PATIENCE,
        restore_best_weights=True
    )


    # ========================================================
    # Training
    # ========================================================

    history = model.fit(
        opt_trainX,
        opt_trainY,

        validation_data=(
            opt_valX,
            opt_valY
        ),

        epochs=MAX_EPOCHS,

        batch_size=batch_size,

        callbacks=[
            early_stop
        ],

        shuffle=False,

        verbose=0
    )


    # ========================================================
    # Validation prediction
    # ========================================================

    val_pred = model.predict(
        opt_valX,
        verbose=0
    ).reshape(-1)


    val_real = opt_valY.reshape(-1)


    # ========================================================
    # 使用 RMSE 當 Optuna objective
    # ========================================================

    val_rmse = np.sqrt(
        mean_squared_error(
            val_real,
            val_pred
        )
    )


    # 額外記錄
    trial.set_user_attr(
        'best_epoch',
        int(
            np.argmin(
                history.history['val_loss']
            ) + 1
        )
    )


    return val_rmse


# ============================================================
# 12. 建立 Optuna Study
# ============================================================

sampler = optuna.samplers.TPESampler(
    seed=SEED,
    multivariate=True
)


study = optuna.create_study(
    direction='minimize',
    sampler=sampler,
    study_name='Summer_1D_CNN_Optuna'
)


# ============================================================
# 13. 開始 Optimization
# ============================================================

print("\n")
print("=" * 70)
print("Starting Optuna optimization")
print("=" * 70)


study.optimize(
    objective,
    n_trials=N_TRIALS,
    gc_after_trial=True,
    show_progress_bar=True
)


# ============================================================
# 14. 最佳結果
# ============================================================

print("\n")
print("=" * 70)
print("BEST OPTUNA RESULT")
print("=" * 70)

print(
    "Best Validation RMSE:",
    study.best_value
)

print("\nBest Parameters:")

for key, value in study.best_params.items():

    print(
        f"{key}: {value}"
    )


print(
    "\nBest Epoch:",
    study.best_trial.user_attrs.get(
        'best_epoch'
    )
)


# ============================================================
# 15. 使用最佳參數重新建立模型
# ============================================================

best_params = study.best_params

best_seq_length = best_params[
    'seqLength'
]


# ============================================================
# 16. 重新建立完整 Train / Test Window
# ============================================================

trainX_final, trainY_final = sliding_window_1d(
    trainSet,
    trainLabel,
    best_seq_length
)

testX_final, testY_final = sliding_window_1d(
    testSet,
    testLabel,
    best_seq_length
)


# ============================================================
# 17. Build Final Best Model
# ============================================================

def build_best_model(
    best_params,
    seq_length,
    feature_count
):

    tf.keras.backend.clear_session()


    model = Sequential()


    model.add(
        Input(
            shape=(
                seq_length,
                feature_count
            )
        )
    )


    activation = best_params[
        'activation'
    ]

    dropout_rate = best_params[
        'dropout_rate'
    ]

    n_conv_layers = best_params[
        'n_conv_layers'
    ]


    # --------------------------------------------------------
    # Conv layers
    # --------------------------------------------------------

    for i in range(
        n_conv_layers
    ):

        model.add(
            Conv1D(
                filters=best_params[
                    f'filters_{i + 1}'
                ],

                kernel_size=best_params[
                    f'kernel_size_{i + 1}'
                ],

                strides=best_params[
                    f'stride_{i + 1}'
                ],

                activation=activation,

                padding='valid'
            )
        )


    # --------------------------------------------------------
    # Pool
    # --------------------------------------------------------

    if best_params[
        'use_pooling'
    ]:

        model.add(
            MaxPooling1D(
                pool_size=best_params[
                    'pooling_size'
                ]
            )
        )


    model.add(
        Flatten()
    )


    # --------------------------------------------------------
    # Dense
    # --------------------------------------------------------

    n_dense_layers = best_params[
        'n_dense_layers'
    ]


    for i in range(
        n_dense_layers
    ):

        model.add(
            Dense(
                best_params[
                    f'dense_units_{i + 1}'
                ],

                activation=activation
            )
        )


        if dropout_rate > 0:

            model.add(
                Dropout(
                    dropout_rate
                )
            )


    model.add(
        Dense(1)
    )


    optimizer = get_optimizer(
        best_params[
            'optimizer'
        ],

        best_params[
            'learning_rate'
        ]
    )


    model.compile(
        optimizer=optimizer,
        loss='mse'
    )


    return model


# ============================================================
# 18. Final Model
# ============================================================

best_model = build_best_model(
    best_params,
    best_seq_length,
    trainX_final.shape[2]
)


best_model.summary()


# ============================================================
# 19. Final Training
# ============================================================

best_epoch = study.best_trial.user_attrs.get(
    'best_epoch'
)


if best_epoch is None:

    best_epoch = MAX_EPOCHS


print(
    "\nFinal training epochs:",
    best_epoch
)


history_final = best_model.fit(
    trainX_final,
    trainY_final,

    epochs=best_epoch,

    batch_size=best_params[
        'batch_size'
    ],

    shuffle=False,

    verbose=1
)


# ============================================================
# 20. Test Prediction
# ============================================================

yhat = best_model.predict(
    testX_final,
    verbose=0
).reshape(-1)


real = testY_final.reshape(-1)


# ============================================================
# 21. Metrics
# ============================================================

mae = mean_absolute_error(
    real,
    yhat
)

rmse = np.sqrt(
    mean_squared_error(
        real,
        yhat
    )
)

r2 = r2_score(
    real,
    yhat
)

test_mse = mean_squared_error(
    real,
    yhat
)


print("\n")
print("=" * 70)
print("FINAL SUMMER TEST RESULT")
print("=" * 70)

print(
    f"MAE      : {mae:.6f}"
)

print(
    f"RMSE     : {rmse:.6f}"
)

print(
    f"R2       : {r2:.6f}"
)

print(
    f"Test MSE : {test_mse:.6f}"
)


# ============================================================
# 22. Results DataFrame
# ============================================================

result_df = pd.DataFrame(
    [
        {
            'Season': season,

            'MAE': mae,

            'RMSE': rmse,

            'R2': r2,

            'Test_MSE': test_mse,

            'SeqLength': best_seq_length,

            'Best_Val_RMSE': study.best_value
        }
    ]
)


print("\n===== Result DataFrame =====")

print(
    result_df
)


# ============================================================
# 23. Plot Real vs Prediction
# ============================================================

plt.figure(
    figsize=(16, 6)
)


plt.plot(
    real,
    label='Real'
)


plt.plot(
    yhat,
    label='Estimated by Optuna 1D-CNN'
)


plt.title(
    'Summer One-hour Ahead Wind Speed - Optuna 1D-CNN'
)

plt.xlabel(
    'Test sample index'
)

plt.ylabel(
    'Wind speed'
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# 24. Optuna Trial History
# ============================================================

trial_df = study.trials_dataframe()

print(
    "\n===== Optuna Trial History ====="
)

print(
    trial_df.head()
)


# ============================================================
# 25. 儲存結果
# ============================================================

trial_df.to_csv(
    'summer_1dcnn_optuna_trials.csv',
    index=False
)


result_df.to_csv(
    'summer_1dcnn_optuna_best_result.csv',
    index=False
)


best_model.save(
    'summer_1dcnn_optuna_best_model.keras'
)


print(
    "\nOptuna trial history saved:"
)

print(
    "summer_1dcnn_optuna_trials.csv"
)

print(
    "\nBest result saved:"
)

print(
    "summer_1dcnn_optuna_best_result.csv"
)

print(
    "\nBest model saved:"
)

print(
    "summer_1dcnn_optuna_best_model.keras"
)

In [ ]:
# 單一條 1D-CNN + Optuna

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import optuna

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    Flatten,
    Dense,
    Dropout
)

from tensorflow.keras.optimizers import (
    Adam,
    RMSprop,
    Nadam
)

from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# 1. Basic Setting
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

season = 'summer'

target_col = 'target_windspeed_1h'

# 建議先跑 50
N_TRIALS = 50

# 每個 Trial 最多訓練 100 epochs
MAX_EPOCHS = 100

# Early stopping
PATIENCE = 10

# Train 裡面最後 20% 當 validation
VALIDATION_RATIO = 0.2


# ============================================================
# 2. Sliding Window
# ============================================================

def sliding_window_1d(X, y, seq_length):

    dataX = []
    dataY = []

    for i in range(
        len(X) - seq_length + 1
    ):

        dataX.append(
            X[i:i + seq_length]
        )

        dataY.append(
            y[i + seq_length - 1]
        )

    return (
        np.array(dataX),
        np.array(dataY).reshape(-1, 1)
    )


# ============================================================
# 3. Load Summer Data
# ============================================================

df_train = season_data[season]['df_train'].copy()
df_test = season_data[season]['df_test'].copy()


print("===== Original Columns =====")
print(df_train.columns.tolist())


# ============================================================
# 4. X / y
# ============================================================

X_train_df = df_train.drop(
    columns=[
        'valid_time',
        target_col
    ]
)

y_train_series = df_train[
    target_col
]


X_test_df = df_test.drop(
    columns=[
        'valid_time',
        target_col
    ]
)

y_test_series = df_test[
    target_col
]


# ============================================================
# 5. Convert Numeric
# ============================================================

X_train_df = X_train_df.apply(
    pd.to_numeric,
    errors='coerce'
)

X_test_df = X_test_df.apply(
    pd.to_numeric,
    errors='coerce'
)

y_train_series = pd.to_numeric(
    y_train_series,
    errors='coerce'
)

y_test_series = pd.to_numeric(
    y_test_series,
    errors='coerce'
)


# ============================================================
# 6. Fill NaN
# ============================================================

train_median = X_train_df.median()

X_train_df = (
    X_train_df
    .fillna(train_median)
    .fillna(0)
)

X_test_df = (
    X_test_df
    .fillna(train_median)
    .fillna(0)
)


y_train_series = y_train_series.fillna(
    y_train_series.median()
)

y_test_series = y_test_series.fillna(
    y_train_series.median()
)


# ============================================================
# 7. Show Features
# ============================================================

print("\n===== CNN Input Features =====")

print(
    X_train_df.columns.tolist()
)

print(
    "\nFeature Count:",
    X_train_df.shape[1]
)


# ============================================================
# 8. MinMax Normalization
# ONLY use Train Min / Max
# ============================================================

train_min = X_train_df.min()
train_max = X_train_df.max()

denom = train_max - train_min

denom[
    denom == 0
] = 1


trainSet = (
    (X_train_df - train_min)
    / denom
).values.astype(np.float32)


testSet = (
    (X_test_df - train_min)
    / denom
).values.astype(np.float32)


trainLabel = (
    y_train_series
    .values
    .astype(np.float32)
)

testLabel = (
    y_test_series
    .values
    .astype(np.float32)
)


# ============================================================
# 9. Optimizer Function
# ============================================================

def get_optimizer(
    optimizer_name,
    learning_rate
):

    if optimizer_name == 'Adam':

        return Adam(
            learning_rate=learning_rate
        )

    elif optimizer_name == 'RMSprop':

        return RMSprop(
            learning_rate=learning_rate
        )

    elif optimizer_name == 'Nadam':

        return Nadam(
            learning_rate=learning_rate
        )

    else:

        raise ValueError(
            f"Unknown optimizer: {optimizer_name}"
        )


# ============================================================
# 10. Optuna Objective
# ============================================================

def objective(trial):

    tf.keras.backend.clear_session()


    # ========================================================
    # Hyperparameters
    # ========================================================

    # Sequence length
    # 1,2,3,...24
    seq_length = trial.suggest_int(
        'seqLength',
        1,
        24
    )


    # Conv1D filters
    # 16,32,48,...128
    filters = trial.suggest_int(
        'filters',
        16,
        128,
        step=16
    )


    # Kernel
    # 1,2,3,4,5
    kernel_size = trial.suggest_int(
        'kernel_size',
        1,
        5
    )


    # Pooling
    # 1,2,3
    pooling_size = trial.suggest_int(
        'pooling_size',
        1,
        3
    )


    # Dense units
    # 16,32,48,...128
    dense_units = trial.suggest_int(
        'dense_units',
        16,
        128,
        step=16
    )


    # Dropout
    dropout_rate = trial.suggest_float(
        'dropout_rate',
        0.0,
        0.3,
        step=0.1
    )


    # Activation
    activation = trial.suggest_categorical(
        'activation',
        [
            'relu',
            'elu',
            'tanh'
        ]
    )


    # Optimizer
    optimizer_name = trial.suggest_categorical(
        'optimizer',
        [
            'Adam',
            'RMSprop',
            'Nadam'
        ]
    )


    # Learning Rate
    learning_rate = trial.suggest_float(
        'learning_rate',
        1e-4,
        5e-3,
        log=True
    )


    # Batch Size
    batch_size = trial.suggest_categorical(
        'batch_size',
        [
            16,
            32,
            64
        ]
    )


    # ========================================================
    # Generate Sliding Window
    # ========================================================

    fullX, fullY = sliding_window_1d(
        trainSet,
        trainLabel,
        seq_length
    )


    # ========================================================
    # Time-Series Train / Validation Split
    #
    # 前 80% = Train
    # 後 20% = Validation
    #
    # 不 Shuffle
    # ========================================================

    split_index = int(
        len(fullX)
        * (1 - VALIDATION_RATIO)
    )


    opt_trainX = fullX[
        :split_index
    ]

    opt_trainY = fullY[
        :split_index
    ]


    opt_valX = fullX[
        split_index:
    ]

    opt_valY = fullY[
        split_index:
    ]


    # ========================================================
    # Build Single 1D-CNN
    # ========================================================

    model = Sequential()


    model.add(
        Input(
            shape=(
                seq_length,
                trainSet.shape[1]
            )
        )
    )


    # --------------------------------------------------------
    # ONLY ONE Conv1D
    # --------------------------------------------------------

    model.add(
        Conv1D(
            filters=filters,
            kernel_size=kernel_size,
            strides=1,
            padding='same',
            activation=activation
        )
    )


    # --------------------------------------------------------
    # Pooling
    #
    # seqLength 很小時避免 pooling size > sequence
    # --------------------------------------------------------

    actual_pooling_size = min(
        pooling_size,
        seq_length
    )


    model.add(
        MaxPooling1D(
            pool_size=actual_pooling_size,
            padding='same'
        )
    )


    # --------------------------------------------------------
    # Flatten
    # --------------------------------------------------------

    model.add(
        Flatten()
    )


    # --------------------------------------------------------
    # Dense
    # --------------------------------------------------------

    model.add(
        Dense(
            dense_units,
            activation=activation
        )
    )


    # --------------------------------------------------------
    # Dropout
    # --------------------------------------------------------

    if dropout_rate > 0:

        model.add(
            Dropout(
                dropout_rate
            )
        )


    # --------------------------------------------------------
    # Output
    # --------------------------------------------------------

    model.add(
        Dense(1)
    )


    # ========================================================
    # Compile
    # ========================================================

    optimizer = get_optimizer(
        optimizer_name,
        learning_rate
    )


    model.compile(
        optimizer=optimizer,
        loss='mse'
    )


    # ========================================================
    # Early Stopping
    # ========================================================

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=PATIENCE,
        restore_best_weights=True
    )


    # ========================================================
    # Train
    # ========================================================

    history = model.fit(
        opt_trainX,
        opt_trainY,

        validation_data=(
            opt_valX,
            opt_valY
        ),

        epochs=MAX_EPOCHS,

        batch_size=batch_size,

        shuffle=False,

        callbacks=[
            early_stop
        ],

        verbose=0
    )


    # ========================================================
    # Validation Prediction
    # ========================================================

    val_prediction = model.predict(
        opt_valX,
        verbose=0
    ).reshape(-1)


    val_real = opt_valY.reshape(-1)


    # ========================================================
    # Validation RMSE
    # Optuna 找「最低 RMSE」
    # ========================================================

    val_rmse = np.sqrt(
        mean_squared_error(
            val_real,
            val_prediction
        )
    )


    # 記錄最佳 epoch
    best_epoch = (
        np.argmin(
            history.history[
                'val_loss'
            ]
        )
        + 1
    )


    trial.set_user_attr(
        'best_epoch',
        int(best_epoch)
    )


    return val_rmse


# ============================================================
# 11. Create Optuna Study
# ============================================================

sampler = optuna.samplers.TPESampler(
    seed=SEED
)


study = optuna.create_study(
    direction='minimize',
    sampler=sampler,
    study_name='Summer_Single_1D_CNN'
)


# ============================================================
# 12. Start Optuna
# ============================================================

print("\n")
print("=" * 70)
print("STARTING SINGLE 1D-CNN OPTUNA")
print("=" * 70)


study.optimize(
    objective,
    n_trials=N_TRIALS,
    gc_after_trial=True,
    show_progress_bar=True
)


# ============================================================
# 13. Show Best Result
# ============================================================

print("\n")
print("=" * 70)
print("BEST OPTUNA RESULT")
print("=" * 70)


print(
    f"Best Validation RMSE: "
    f"{study.best_value:.6f}"
)


print(
    "\nBest Parameters:"
)


for key, value in study.best_params.items():

    print(
        f"{key}: {value}"
    )


best_epoch = study.best_trial.user_attrs[
    'best_epoch'
]


print(
    f"\nBest Epoch: {best_epoch}"
)


# ============================================================
# 14. Best Parameters
# ============================================================

best_params = study.best_params

best_seq_length = best_params[
    'seqLength'
]


# ============================================================
# 15. Generate Final Train / Test Windows
# ============================================================

trainX_final, trainY_final = sliding_window_1d(
    trainSet,
    trainLabel,
    best_seq_length
)


testX_final, testY_final = sliding_window_1d(
    testSet,
    testLabel,
    best_seq_length
)


# ============================================================
# 16. Build Final Best Model
# ============================================================

def build_best_single_1d_model(
    params,
    seq_length,
    feature_count
):

    tf.keras.backend.clear_session()


    model = Sequential()


    model.add(
        Input(
            shape=(
                seq_length,
                feature_count
            )
        )
    )


    # ========================================================
    # Single Conv1D
    # ========================================================

    model.add(
        Conv1D(
            filters=params[
                'filters'
            ],

            kernel_size=params[
                'kernel_size'
            ],

            strides=1,

            padding='same',

            activation=params[
                'activation'
            ]
        )
    )


    # ========================================================
    # Pooling
    # ========================================================

    actual_pooling_size = min(
        params[
            'pooling_size'
        ],
        seq_length
    )


    model.add(
        MaxPooling1D(
            pool_size=actual_pooling_size,
            padding='same'
        )
    )


    # ========================================================
    # Flatten
    # ========================================================

    model.add(
        Flatten()
    )


    # ========================================================
    # Dense
    # ========================================================

    model.add(
        Dense(
            params[
                'dense_units'
            ],

            activation=params[
                'activation'
            ]
        )
    )


    # ========================================================
    # Dropout
    # ========================================================

    if params[
        'dropout_rate'
    ] > 0:

        model.add(
            Dropout(
                params[
                    'dropout_rate'
                ]
            )
        )


    # ========================================================
    # Output
    # ========================================================

    model.add(
        Dense(1)
    )


    # ========================================================
    # Optimizer
    # ========================================================

    optimizer = get_optimizer(
        params[
            'optimizer'
        ],

        params[
            'learning_rate'
        ]
    )


    model.compile(
        optimizer=optimizer,
        loss='mse'
    )


    return model


# ============================================================
# 17. Create Final Model
# ============================================================

best_model = build_best_single_1d_model(
    best_params,
    best_seq_length,
    trainX_final.shape[2]
)


print("\n")
print("=" * 70)
print("FINAL BEST MODEL")
print("=" * 70)


best_model.summary()


# ============================================================
# 18. Retrain Using Full Training Set
#
# 使用 Optuna 找到的 best_epoch
# ============================================================

print(
    f"\nFinal Training Epochs: "
    f"{best_epoch}"
)


history_final = best_model.fit(
    trainX_final,
    trainY_final,

    epochs=best_epoch,

    batch_size=best_params[
        'batch_size'
    ],

    shuffle=False,

    verbose=1
)


# ============================================================
# 19. Final Test Prediction
# ============================================================

yhat = best_model.predict(
    testX_final,
    verbose=0
).reshape(-1)


real = testY_final.reshape(-1)


# ============================================================
# 20. Metrics
# ============================================================

mae = mean_absolute_error(
    real,
    yhat
)


rmse = np.sqrt(
    mean_squared_error(
        real,
        yhat
    )
)


r2 = r2_score(
    real,
    yhat
)


test_mse = mean_squared_error(
    real,
    yhat
)


print("\n")
print("=" * 70)
print("FINAL TEST RESULT")
print("=" * 70)


print(
    f"MAE      : {mae:.6f}"
)

print(
    f"RMSE     : {rmse:.6f}"
)

print(
    f"R2       : {r2:.6f}"
)

print(
    f"Test MSE : {test_mse:.6f}"
)


# ============================================================
# 21. Result DataFrame
# ============================================================

result_df = pd.DataFrame(
    [
        {
            'Architecture':
                'Single 1D-CNN',

            'Season':
                season,

            'MAE':
                mae,

            'RMSE':
                rmse,

            'R2':
                r2,

            'Test_MSE':
                test_mse,

            'Best_Val_RMSE':
                study.best_value,

            'SeqLength':
                best_seq_length,

            'Filters':
                best_params[
                    'filters'
                ],

            'Kernel_Size':
                best_params[
                    'kernel_size'
                ],

            'Pooling_Size':
                best_params[
                    'pooling_size'
                ],

            'Dense_Units':
                best_params[
                    'dense_units'
                ],

            'Dropout':
                best_params[
                    'dropout_rate'
                ],

            'Activation':
                best_params[
                    'activation'
                ],

            'Optimizer':
                best_params[
                    'optimizer'
                ],

            'Learning_Rate':
                best_params[
                    'learning_rate'
                ],

            'Batch_Size':
                best_params[
                    'batch_size'
                ],

            'Best_Epoch':
                best_epoch
        }
    ]
)


print("\n===== FINAL RESULT =====")

print(
    result_df.to_string(
        index=False
    )
)


# ============================================================
# 22. Plot
# ============================================================

plt.figure(
    figsize=(16, 6)
)


plt.plot(
    real,
    label='Real'
)


plt.plot(
    yhat,
    label='Single 1D-CNN + Optuna'
)


plt.title(
    'Summer One-hour Ahead Wind Speed '
    '- Single 1D-CNN + Optuna'
)


plt.xlabel(
    'Test Sample Index'
)


plt.ylabel(
    'Wind Speed'
)


plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# 23. Save
# ============================================================

trial_df = study.trials_dataframe()


trial_df.to_csv(
    'single_1dcnn_optuna_trials.csv',
    index=False
)


result_df.to_csv(
    'single_1dcnn_optuna_result.csv',
    index=False
)


best_model.save(
    'single_1dcnn_optuna_best_model.keras'
)


print("\n===== Saved =====")

print(
    "single_1dcnn_optuna_trials.csv"
)

print(
    "single_1dcnn_optuna_result.csv"
)

print(
    "single_1dcnn_optuna_best_model.keras"
)

In [ ]:
# 1D//1D//1D

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    MaxPooling1D,
    GlobalMaxPooling1D,
    Concatenate,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ============================================================
# 1. 固定亂數種子
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 2. 基本設定
# ============================================================

SEQ_LENGTH = 6
TRAIN_RATIO = 0.8

EPOCHS = 100
BATCH_SIZE = 32

FILTERS = 100

TARGET_COLUMN = "windspeed_120"

season_files = {
    "Spring": "clearspring.csv",
    "Summer": "clearsummer.csv",
    "Autumn": "clearautumn.csv",
    "Winter": "clearwinter.csv"
}


# ============================================================
# 3. 建立時間序列 Window
#
# X:
# 過去 seq_length 小時的氣象資料
#
# y:
# 對應下一小時 windspeed_120
# ============================================================

def create_sequences(X, y, seq_length):

    X_seq = []
    y_seq = []

    for i in range(len(X) - seq_length + 1):

        X_seq.append(
            X[i:i + seq_length]
        )

        y_seq.append(
            y[i + seq_length - 1]
        )

    return np.array(X_seq), np.array(y_seq)


# ============================================================
# 4. Parallel 1D-CNN
# ============================================================

def build_parallel_1dcnn(input_shape):

    inputs = Input(
        shape=input_shape,
        name="Weather_Input"
    )

    # --------------------------------------------------------
    # Branch 1
    # kernel_size = 2
    # 看非常短期的變化
    # --------------------------------------------------------

    branch_1 = Conv1D(
        filters=FILTERS,
        kernel_size=2,
        activation="relu",
        padding="same",
        name="Conv1D_Kernel_2"
    )(inputs)

    branch_1 = MaxPooling1D(
        pool_size=2,
        name="Pool_Kernel_2"
    )(branch_1)

    branch_1 = GlobalMaxPooling1D(
        name="GlobalPool_2"
    )(branch_1)


    # --------------------------------------------------------
    # Branch 2
    # kernel_size = 3
    # 看中短期變化
    # --------------------------------------------------------

    branch_2 = Conv1D(
        filters=FILTERS,
        kernel_size=3,
        activation="relu",
        padding="same",
        name="Conv1D_Kernel_3"
    )(inputs)

    branch_2 = MaxPooling1D(
        pool_size=2,
        name="Pool_Kernel_3"
    )(branch_2)

    branch_2 = GlobalMaxPooling1D(
        name="GlobalPool_3"
    )(branch_2)


    # --------------------------------------------------------
    # Branch 3
    # kernel_size = 5
    # 看比較長的變化
    # --------------------------------------------------------

    branch_3 = Conv1D(
        filters=FILTERS,
        kernel_size=5,
        activation="relu",
        padding="same",
        name="Conv1D_Kernel_5"
    )(inputs)

    branch_3 = MaxPooling1D(
        pool_size=2,
        name="Pool_Kernel_5"
    )(branch_3)

    branch_3 = GlobalMaxPooling1D(
        name="GlobalPool_5"
    )(branch_3)


    # ========================================================
    # 合併三條 CNN
    # ========================================================

    merged = Concatenate(
        name="Concatenate_Features"
    )([
        branch_1,
        branch_2,
        branch_3
    ])


    # ========================================================
    # Dense
    # ========================================================

    x = Dense(
        48,
        activation="relu",
        name="Dense_48"
    )(merged)

    x = Dropout(
        0.2,
        name="Dropout"
    )(x)


    # ========================================================
    # Regression output
    # ========================================================

    output = Dense(
        1,
        name="Predicted_Windspeed"
    )(x)


    model = Model(
        inputs=inputs,
        outputs=output,
        name="Parallel_1D_CNN"
    )

    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae"]
    )

    return model


# ============================================================
# 5. 儲存各季結果
# ============================================================

results = []

predictions = {}


# ============================================================
# 6. 四季開始訓練
# ============================================================

for season, file_path in season_files.items():

    print("\n")
    print("=" * 70)
    print(f"Processing Season: {season}")
    print("=" * 70)


    # ========================================================
    # Load CSV
    # ========================================================

    df = pd.read_csv(file_path)

    print(f"Original rows: {len(df)}")


    # ========================================================
    # 檢查 target
    # ========================================================

    if TARGET_COLUMN not in df.columns:

        raise ValueError(
            f"{file_path} 找不到欄位：{TARGET_COLUMN}"
        )


    # ========================================================
    # 建立下一小時 Target
    #
    # 現在 t
    # ↓
    # 預測 t+1 windspeed_120
    # ========================================================

    df["target_windspeed_1h"] = (
        df[TARGET_COLUMN]
        .shift(-1)
    )


    # 最後一筆沒有下一小時資料
    df = df.dropna(
        subset=["target_windspeed_1h"]
    ).reset_index(drop=True)


    # ========================================================
    # 只使用 numeric features
    # ========================================================

    numeric_columns = (
        df
        .select_dtypes(include=[np.number])
        .columns
        .tolist()
    )


    # Target 不可以放進 input
    feature_columns = [
        col
        for col in numeric_columns
        if col != "target_windspeed_1h"
    ]


    print("\nFeatures:")
    print(feature_columns)

    print("\nNumber of features:")
    print(len(feature_columns))


    # ========================================================
    # X / y
    # ========================================================

    X = df[feature_columns].values

    y = df[
        ["target_windspeed_1h"]
    ].values


    # ========================================================
    # Train / Test split
    #
    # 時序資料不能 shuffle
    # ========================================================

    split_index = int(
        len(df) * TRAIN_RATIO
    )


    X_train_raw = X[:split_index]
    X_test_raw = X[split_index:]

    y_train_raw = y[:split_index]
    y_test_raw = y[split_index:]


    print("\nTrain raw shape:")
    print(X_train_raw.shape)

    print("Test raw shape:")
    print(X_test_raw.shape)


    # ========================================================
    # Scaler
    #
    # 非常重要：
    # scaler 只能 fit train
    # ========================================================

    X_scaler = StandardScaler()

    X_scaler.fit(
        X_train_raw
    )


    X_train_scaled = X_scaler.transform(
        X_train_raw
    )

    X_test_scaled = X_scaler.transform(
        X_test_raw
    )


    # Target scaler
    y_scaler = StandardScaler()

    y_scaler.fit(
        y_train_raw
    )


    y_train_scaled = y_scaler.transform(
        y_train_raw
    )

    y_test_scaled = y_scaler.transform(
        y_test_raw
    )


    # ========================================================
    # 建立 Sequence
    # ========================================================

    X_train_seq, y_train_seq = create_sequences(
        X_train_scaled,
        y_train_scaled,
        SEQ_LENGTH
    )

    X_test_seq, y_test_seq = create_sequences(
        X_test_scaled,
        y_test_scaled,
        SEQ_LENGTH
    )


    print("\nSequence shapes:")

    print(
        "X_train:",
        X_train_seq.shape
    )

    print(
        "y_train:",
        y_train_seq.shape
    )

    print(
        "X_test:",
        X_test_seq.shape
    )

    print(
        "y_test:",
        y_test_seq.shape
    )


    # ========================================================
    # 檢查 NaN
    # ========================================================

    print("\nNaN Check:")

    print(
        "X_train NaN:",
        np.isnan(X_train_seq).sum()
    )

    print(
        "y_train NaN:",
        np.isnan(y_train_seq).sum()
    )

    print(
        "X_test NaN:",
        np.isnan(X_test_seq).sum()
    )

    print(
        "y_test NaN:",
        np.isnan(y_test_seq).sum()
    )


    # ========================================================
    # 建立 Parallel 1D CNN
    # ========================================================

    model = build_parallel_1dcnn(
        input_shape=(
            X_train_seq.shape[1],
            X_train_seq.shape[2]
        )
    )


    # 只第一次顯示 architecture
    if season == "Spring":

        print("\n")
        print("=" * 70)
        print("Parallel 1D-CNN Architecture")
        print("=" * 70)

        model.summary()


    # ========================================================
    # Early stopping
    # ========================================================

    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True
    )


    # ========================================================
    # Training
    # ========================================================

    history = model.fit(

        X_train_seq,
        y_train_seq,

        validation_split=0.2,

        epochs=EPOCHS,

        batch_size=BATCH_SIZE,

        shuffle=False,

        callbacks=[
            early_stopping
        ],

        verbose=1
    )


    # ========================================================
    # Predict
    # ========================================================

    y_pred_scaled = model.predict(
        X_test_seq
    )


    # ========================================================
    # inverse transform
    # ========================================================

    y_pred = y_scaler.inverse_transform(
        y_pred_scaled
    ).flatten()

    y_true = y_scaler.inverse_transform(
        y_test_seq
    ).flatten()


    # ========================================================
    # Metrics
    # ========================================================

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    r2 = r2_score(
        y_true,
        y_pred
    )


    print("\n")
    print("=" * 50)
    print(f"{season} Results")
    print("=" * 50)

    print(
        f"MAE  : {mae:.4f}"
    )

    print(
        f"RMSE : {rmse:.4f}"
    )

    print(
        f"R²   : {r2:.4f}"
    )


    # ========================================================
    # Save results
    # ========================================================

    results.append({

        "Season": season,

        "MAE": mae,

        "RMSE": rmse,

        "R2": r2
    })


    predictions[season] = {

        "actual": y_true,

        "predicted": y_pred
    }


# ============================================================
# 7. Results Table
# ============================================================

results_df = pd.DataFrame(
    results
)

print("\n")
print("=" * 70)
print("Final Parallel 1D-CNN Results")
print("=" * 70)

print(
    results_df.to_string(
        index=False
    )
)


# ============================================================
# 8. 四季 Actual vs Prediction
# ============================================================

fig, axes = plt.subplots(
    2,
    2,
    figsize=(16, 10)
)

axes = axes.flatten()


for i, season in enumerate(
    season_files.keys()
):

    actual = predictions[
        season
    ]["actual"]

    predicted = predictions[
        season
    ]["predicted"]


    axes[i].plot(
        actual,
        label="Actual"
    )

    axes[i].plot(
        predicted,
        label="Predicted"
    )


    axes[i].set_title(
        f"{season} - Parallel 1D-CNN"
    )

    axes[i].set_xlabel(
        "Time Step"
    )

    axes[i].set_ylabel(
        "Windspeed 120m"
    )

    axes[i].legend()

    axes[i].grid(
        alpha=0.3
    )


plt.tight_layout()

plt.show()


# ============================================================
# 9. 儲存指標
# ============================================================

results_df.to_csv(
    "parallel_1dcnn_results.csv",
    index=False
)

print(
    "\nResults saved to parallel_1dcnn_results.csv"
)

In [ ]:
# 1D//2D

In [ ]:
# ============================================================
# Parallel 1D-CNN + 2D-CNN
# Summer Windspeed Prediction
# 預測下一小時 windspeed_120
# ============================================================

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    Conv2D,
    MaxPooling1D,
    MaxPooling2D,
    Flatten,
    Dense,
    Concatenate,
    Reshape
)

from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# 1. 固定 Random Seed
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# 2. Hyper Parameters
# ============================================================

SEQ_LENGTH = 6

FILTERS = 100

KERNEL_1D = 2
KERNEL_2D = (2, 2)

POOL_1D = 2
POOL_2D = (2, 2)

EPOCHS = 100
BATCH_SIZE = 32

TRAIN_RATIO = 0.8

TARGET_COLUMN = "windspeed_120"

FILE_PATH = "clearsummer.csv"

# ============================================================
# Quantile 設定
# ============================================================

QUANTILE = 0.5


# ============================================================
# 3. Sliding Window
# ============================================================

def create_sequences(X, y, seq_length):

    dataX = []
    dataY = []

    for i in range(
        len(X) - seq_length + 1
    ):

        # 過去 seq_length 個時間點
        dataX.append(
            X[i:i + seq_length]
        )

        # 對應 target
        dataY.append(
            y[i + seq_length - 1]
        )

    return (
        np.array(dataX),
        np.array(dataY).reshape(-1, 1)
    )

# ============================================================
# Quantile Loss
# ============================================================

def quantile_loss(q):
    def loss(y_true, y_pred):

        error = y_true - y_pred

        return tf.reduce_mean(
            tf.maximum(
                q * error,
                (q - 1) * error
            )
        )

    return loss

# ============================================================
# 4. 建立 Parallel 1D-CNN + 2D-CNN
# ============================================================

def build_parallel_1d_2d_cnn(
    seq_length,
    feature_count
):

    # ========================================================
    # 共用 Input
    #
    # Shape:
    # (seq_length, feature_count)
    #
    # 例如：
    # (6, 12)
    # ========================================================

    inputs = Input(
        shape=(
            seq_length,
            feature_count
        ),
        name="Weather_Input"
    )


    # ========================================================
    # Branch 1
    # 1D-CNN
    #
    # 主要學習時間序列局部特徵
    # ========================================================

    branch_1d = Conv1D(
        filters=FILTERS,
        kernel_size=KERNEL_1D,
        strides=1,
        activation="relu",
        padding="same",
        name="Conv1D_Branch"
    )(inputs)


    branch_1d = MaxPooling1D(
        pool_size=POOL_1D,
        name="MaxPooling1D"
    )(branch_1d)


    branch_1d = Flatten(
        name="Flatten_1D"
    )(branch_1d)


    # ========================================================
    # Branch 2
    # 2D-CNN
    #
    # Conv2D 需要：
    #
    # (height, width, channel)
    #
    # 所以把：
    #
    # (6, features)
    #
    # 轉成：
    #
    # (6, features, 1)
    # ========================================================

    branch_2d = Reshape(
        (
            seq_length,
            feature_count,
            1
        ),
        name="Reshape_for_2D"
    )(inputs)


    branch_2d = Conv2D(
        filters=FILTERS,
        kernel_size=KERNEL_2D,
        strides=(1, 1),
        activation="relu",
        padding="same",
        name="Conv2D_Branch"
    )(branch_2d)


    branch_2d = MaxPooling2D(
        pool_size=POOL_2D,
        name="MaxPooling2D"
    )(branch_2d)


    branch_2d = Flatten(
        name="Flatten_2D"
    )(branch_2d)


    # ========================================================
    # Feature Fusion
    #
    # 1D Feature
    #      +
    # 2D Feature
    # ========================================================

    merged = Concatenate(
        name="Concatenate_1D_2D"
    )([
        branch_1d,
        branch_2d
    ])


    # ========================================================
    # Dense Layer
    # ========================================================

    dense = Dense(
        48,
        activation="relu",
        name="Dense_48"
    )(merged)


    # ========================================================
    # Output
    #
    # Regression
    # 預測下一小時 windspeed_120
    # ========================================================

    output = Dense(
        1,
        name="Predicted_Windspeed"
    )(dense)


    # ========================================================
    # Model
    # ========================================================

    model.compile(
    optimizer="adam",
    loss=quantile_loss(QUANTILE),
    metrics=["mae"]
)

    return model


# ============================================================
# 5. Load Summer Dataset
# ============================================================

print("\n======================================")
print("Loading Summer Dataset")
print("======================================")

df = pd.read_csv(
    FILE_PATH
)

print(
    "Original rows:",
    len(df)
)


# ============================================================
# 6. 檢查 Target Column
# ============================================================

if TARGET_COLUMN not in df.columns:

    raise ValueError(
        f"找不到欄位：{TARGET_COLUMN}"
    )


# ============================================================
# 7. 建立下一小時 Target
#
# t 時間點的資料
#       ↓
# 預測
#       ↓
# t+1 的 windspeed_120
# ============================================================

df["target_windspeed_1h"] = (
    df[TARGET_COLUMN]
    .shift(-1)
)


# 最後一筆沒有 t+1
df = df.dropna(
    subset=["target_windspeed_1h"]
).reset_index(drop=True)


print(
    "Rows after target shift:",
    len(df)
)


# ============================================================
# 8. 選擇 Numeric Features
# ============================================================

numeric_columns = (
    df
    .select_dtypes(include=[np.number])
    .columns
    .tolist()
)


# target_windspeed_1h 不可以放進 Input
feature_columns = [

    col

    for col in numeric_columns

    if col != "target_windspeed_1h"
]


print("\n======================================")
print("Input Features")
print("======================================")

print(
    feature_columns
)

print(
    "\nFeature Count:",
    len(feature_columns)
)


# ============================================================
# 9. 建立 X / y
# ============================================================

X = df[
    feature_columns
].values


y = df[
    ["target_windspeed_1h"]
].values


print("\nX shape:", X.shape)
print("y shape:", y.shape)


# ============================================================
# 10. Train / Test Split
#
# 時間序列：
# 不可以 Random Shuffle
#
# 前 80% = Train
# 後 20% = Test
# ============================================================

split_index = int(
    len(df) * TRAIN_RATIO
)


X_train_raw = X[
    :split_index
]

X_test_raw = X[
    split_index:
]


y_train_raw = y[
    :split_index
]

y_test_raw = y[
    split_index:
]


print("\n======================================")
print("Train / Test")
print("======================================")

print(
    "X Train:",
    X_train_raw.shape
)

print(
    "X Test :",
    X_test_raw.shape
)


# ============================================================
# 11. StandardScaler
#
# 非常重要：
#
# Scaler 只能 FIT Train
#
# Test 只能 Transform
#
# 避免 Data Leakage
# ============================================================

X_scaler = StandardScaler()


X_scaler.fit(
    X_train_raw
)


X_train_scaled = X_scaler.transform(
    X_train_raw
)


X_test_scaled = X_scaler.transform(
    X_test_raw
)


# ============================================================
# Target Scaler
# ============================================================

y_scaler = StandardScaler()


y_scaler.fit(
    y_train_raw
)


y_train_scaled = y_scaler.transform(
    y_train_raw
)


y_test_scaled = y_scaler.transform(
    y_test_raw
)


# ============================================================
# 12. 建立 Sliding Window
# ============================================================

X_train_seq, y_train_seq = create_sequences(

    X_train_scaled,

    y_train_scaled,

    SEQ_LENGTH
)


X_test_seq, y_test_seq = create_sequences(

    X_test_scaled,

    y_test_scaled,

    SEQ_LENGTH
)


print("\n======================================")
print("Sequence Shape")
print("======================================")

print(
    "X_train_seq:",
    X_train_seq.shape
)

print(
    "y_train_seq:",
    y_train_seq.shape
)

print(
    "X_test_seq :",
    X_test_seq.shape
)

print(
    "y_test_seq :",
    y_test_seq.shape
)


# ============================================================
# 13. NaN Check
# ============================================================

print("\n======================================")
print("NaN Check")
print("======================================")

print(
    "X Train NaN:",
    np.isnan(X_train_seq).sum()
)

print(
    "y Train NaN:",
    np.isnan(y_train_seq).sum()
)

print(
    "X Test NaN:",
    np.isnan(X_test_seq).sum()
)

print(
    "y Test NaN:",
    np.isnan(y_test_seq).sum()
)


# ============================================================
# 14. 建立 Parallel Model
# ============================================================

model = build_parallel_1d_2d_cnn(

    seq_length=SEQ_LENGTH,

    feature_count=X_train_seq.shape[2]
)


# ============================================================
# 15. Model Summary
# ============================================================

print("\n======================================")
print("Parallel 1D + 2D CNN Architecture")
print("======================================")

model.summary()


# ============================================================
# 16. Early Stopping
# ============================================================

early_stopping = EarlyStopping(

    monitor="val_loss",

    patience=10,

    restore_best_weights=True
)


# ============================================================
# 17. Training
# ============================================================

print("\n======================================")
print("Training")
print("======================================")

history = model.fit(

    X_train_seq,

    y_train_seq,

    validation_split=0.2,

    epochs=EPOCHS,

    batch_size=BATCH_SIZE,

    shuffle=False,

    callbacks=[
        early_stopping
    ],

    verbose=1
)


# ============================================================
# 18. Predict
# ============================================================

y_pred_scaled = model.predict(
    X_test_seq
)


# ============================================================
# 19. Inverse Transform
#
# 還原成原本 windspeed 單位
# ============================================================

y_pred = y_scaler.inverse_transform(
    y_pred_scaled
).flatten()


y_true = y_scaler.inverse_transform(
    y_test_seq
).flatten()


# ============================================================
# 20. Evaluation
# ============================================================

mae = mean_absolute_error(
    y_true,
    y_pred
)


rmse = np.sqrt(

    mean_squared_error(
        y_true,
        y_pred
    )
)


r2 = r2_score(
    y_true,
    y_pred
)


print("\n======================================")
print(f"Summer Parallel 1D + 2D CNN")
print(f"Quantile Loss q = {QUANTILE}")
print("======================================")

print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")

print(
    f"MAE  : {mae:.4f}"
)

print(
    f"RMSE : {rmse:.4f}"
)

print(
    f"R²   : {r2:.4f}"
)


# ============================================================
# 21. Actual vs Predicted
# ============================================================

plt.figure(
    figsize=(14, 6)
)


plt.plot(
    y_true,
    label="Actual"
)


plt.plot(
    y_pred,
    label="Predicted"
)


plt.title(
    "Summer - Parallel 1D-CNN + 2D-CNN"
)


plt.xlabel(
    "Time Step"
)


plt.ylabel(
    "Windspeed 120m"
)


plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()


# ============================================================
# 22. Training Loss
# ============================================================

plt.figure(
    figsize=(10, 5)
)


plt.plot(
    history.history["loss"],
    label="Training Loss"
)


plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)


plt.title(
    f"Parallel 1D + 2D CNN - Quantile Loss (q={QUANTILE})"
)


plt.xlabel(
    "Epoch"
)


plt.ylabel(
    "Quantile Loss"
)


plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()


# ============================================================
# 23. 儲存結果
# ============================================================

results = pd.DataFrame({

    "Actual": y_true,

    "Predicted": y_pred

})


results.to_csv(
    f"summer_parallel_1d_2d_quantile_{QUANTILE}_prediction.csv",
    index=False
)


metrics = pd.DataFrame({

    "Model": [
        "Parallel 1D + 2D CNN"
    ],

    "MAE": [
        mae
    ],

    "RMSE": [
        rmse
    ],

    "R2": [
        r2
    ]

})


metrics.to_csv(
    f"summer_parallel_1d_2d_quantile_{QUANTILE}_metrics.csv",
    index=False
)


print("\nPrediction saved:")
print(
    "summer_parallel_1d_2d_prediction.csv"
)

print("\nMetrics saved:")
print(
    "summer_parallel_1d_2d_metrics.csv"
)

In [ ]:
# 先 1D 再 2D
# 串聯 / Sequential Hybrid CNN

In [ ]:
# ============================================================
# 1D-CNN -> 2D-CNN
# Summer Wind Speed Prediction
# Multi-Quantile Loss
#
# q0.1 = Lower Bound
# q0.5 = Median Prediction
# q0.9 = Upper Bound
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    Conv1D,
    Conv2D,
    MaxPooling2D,
    Reshape,
    Flatten,
    Dense,
    Dropout
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# Hyper Parameters
# ============================================================

seqLength = 6

# -------------------------
# 1D-CNN
# -------------------------

filters_1d_1 = 100
filters_1d_2 = 64
kernel_size_1d = 2


# -------------------------
# 2D-CNN
# -------------------------

filters_2d_1 = 64
filters_2d_2 = 32

kernel_size_2d_1 = (2, 2)
kernel_size_2d_2 = (2, 2)

pooling_size_2d = (2, 2)


# -------------------------
# Training
# -------------------------

epochs = 100
batch_size = 32


# -------------------------
# Quantiles
# -------------------------

quantiles = [0.1, 0.5, 0.9]


# -------------------------
# Target
# -------------------------

target_col = 'target_windspeed_1h'


# ============================================================
# Multi Quantile Loss
# ============================================================

def multi_quantile_loss(quantiles):

    q = tf.constant(
        quantiles,
        dtype=tf.float32
    )

    def loss(y_true, y_pred):

        # y_true:
        # (batch, 1)

        # y_pred:
        # (batch, 3)
        #
        # column 0 = q0.1
        # column 1 = q0.5
        # column 2 = q0.9

        error = y_true - y_pred

        quantile_losses = tf.maximum(
            q * error,
            (q - 1.0) * error
        )

        return tf.reduce_mean(
            quantile_losses
        )

    return loss


# ============================================================
# Sliding Window
# ============================================================

def sliding_window(X, y, seq_length):

    dataX = []
    dataY = []

    for i in range(
        0,
        len(X) - seq_length + 1
    ):

        dataX.append(
            X[i:i + seq_length]
        )

        dataY.append(
            y[i + seq_length - 1]
        )

    return (
        np.array(dataX),
        np.array(dataY).reshape(-1, 1)
    )


# ============================================================
# Build
# 1D-CNN -> 2D-CNN -> Multi Quantile Output
# ============================================================

def build_1d_2d_cnn(
    seq_length,
    feature_count
):

    model = Sequential()


    # ========================================================
    # Input
    # ========================================================

    model.add(
        Input(
            shape=(
                seq_length,
                feature_count
            )
        )
    )


    # ========================================================
    # 1D-CNN
    # ========================================================

    model.add(
        Conv1D(
            filters=filters_1d_1,
            kernel_size=kernel_size_1d,
            activation='relu',
            padding='same'
        )
    )


    model.add(
        Conv1D(
            filters=filters_1d_2,
            kernel_size=kernel_size_1d,
            activation='relu',
            padding='same'
        )
    )


    # ========================================================
    # Reshape
    #
    # (batch, 6, 64)
    #
    # ↓
    #
    # (batch, 6, 64, 1)
    #
    # 讓 2D-CNN 可以接收
    # ========================================================

    model.add(
        Reshape(
            (
                seq_length,
                filters_1d_2,
                1
            )
        )
    )


    # ========================================================
    # 2D-CNN
    # ========================================================

    model.add(
        Conv2D(
            filters=filters_2d_1,
            kernel_size=kernel_size_2d_1,
            activation='relu',
            padding='same'
        )
    )


    model.add(
        Conv2D(
            filters=filters_2d_2,
            kernel_size=kernel_size_2d_2,
            activation='relu',
            padding='same'
        )
    )


    model.add(
        MaxPooling2D(
            pool_size=pooling_size_2d
        )
    )


    # ========================================================
    # Regression
    # ========================================================

    model.add(
        Flatten()
    )


    model.add(
        Dense(
            48,
            activation='relu'
        )
    )


    model.add(
        Dropout(0.2)
    )


    # ========================================================
    # IMPORTANT
    #
    # 原本：
    #
    # Dense(1)
    #
    # 現在：
    #
    # Dense(3)
    #
    # 3個輸出分別是：
    #
    # q0.1
    # q0.5
    # q0.9
    # ========================================================

    model.add(
        Dense(
            len(quantiles)
        )
    )


    # ========================================================
    # Compile
    # ========================================================

    model.compile(

        optimizer='adam',

        loss=multi_quantile_loss(
            quantiles
        )

    )

    return model


# ============================================================
# Summer Data
# ============================================================

season = 'summer'

df_train = (
    season_data[season]['df_train']
    .copy()
)

df_test = (
    season_data[season]['df_test']
    .copy()
)


# ============================================================
# X / y
# ============================================================

X_train = df_train.drop(
    columns=[
        'valid_time',
        target_col
    ]
)

y_train = df_train[
    target_col
]


X_test = df_test.drop(
    columns=[
        'valid_time',
        target_col
    ]
)

y_test = df_test[
    target_col
]


# ============================================================
# Convert Numeric
# ============================================================

X_train = X_train.apply(
    pd.to_numeric,
    errors='coerce'
)

X_test = X_test.apply(
    pd.to_numeric,
    errors='coerce'
)

y_train = pd.to_numeric(
    y_train,
    errors='coerce'
)

y_test = pd.to_numeric(
    y_test,
    errors='coerce'
)


# ============================================================
# Handle NaN
# ============================================================

X_train = (
    X_train
    .fillna(
        X_train.median()
    )
    .fillna(0)
)


X_test = (
    X_test
    .fillna(
        X_train.median()
    )
    .fillna(0)
)


y_train = y_train.fillna(
    y_train.median()
)

y_test = y_test.fillna(
    y_train.median()
)


# ============================================================
# Min-Max Normalization
#
# 只使用 Training Data 的 min / max
# 避免 Data Leakage
# ============================================================

train_min = X_train.min()
train_max = X_train.max()

denom = (
    train_max - train_min
)

denom[
    denom == 0
] = 1


trainSet = (
    (X_train - train_min)
    / denom
).values.astype(
    np.float32
)


testSet = (
    (X_test - train_min)
    / denom
).values.astype(
    np.float32
)


trainLabel = (
    y_train
    .values
    .astype(np.float32)
)

testLabel = (
    y_test
    .values
    .astype(np.float32)
)


# ============================================================
# Sliding Window
# ============================================================

trainX, trainY = sliding_window(
    trainSet,
    trainLabel,
    seqLength
)


testX, testY = sliding_window(
    testSet,
    testLabel,
    seqLength
)


# ============================================================
# Check Shape
# ============================================================

print(
    "\n=============================="
)

print(
    "Data Shape"
)

print(
    "=============================="
)

print(
    "trainX :",
    trainX.shape
)

print(
    "trainY :",
    trainY.shape
)

print(
    "testX  :",
    testX.shape
)

print(
    "testY  :",
    testY.shape
)

print(
    "Feature count :",
    trainX.shape[2]
)


# ============================================================
# Build Model
# ============================================================

model = build_1d_2d_cnn(
    seqLength,
    trainX.shape[2]
)


# ============================================================
# Model Summary
# ============================================================

print(
    "\n======================================"
)

print(
    "1D-CNN -> 2D-CNN Multi-Quantile Model"
)

print(
    "======================================"
)

model.summary()


# ============================================================
# Train
# ============================================================

history = model.fit(

    trainX,

    trainY,

    epochs=epochs,

    batch_size=batch_size,

    validation_split=0.2,

    shuffle=False,

    verbose=1

)


# ============================================================
# Prediction
# ============================================================

prediction = model.predict(
    testX,
    verbose=0
)


# ============================================================
# Separate Quantiles
# ============================================================

pred_q10 = prediction[:, 0]

pred_q50 = prediction[:, 1]

pred_q90 = prediction[:, 2]


real = testY.reshape(-1)


# ============================================================
# Evaluation
#
# q0.5 = Median
#
# 使用 q0.5 作為主要 point prediction
# ============================================================

mae = mean_absolute_error(
    real,
    pred_q50
)


rmse = np.sqrt(
    mean_squared_error(
        real,
        pred_q50
    )
)


r2 = r2_score(
    real,
    pred_q50
)


# ============================================================
# Quantile Range Evaluation
# ============================================================

# q90 - q10
quantile_width = (
    pred_q90 - pred_q10
)


mean_quantile_width = np.mean(
    quantile_width
)


# ============================================================
# Coverage
#
# 真實值有多少比例落在 q10 ~ q90 中間
# ============================================================

coverage = np.mean(

    (real >= pred_q10)

    &

    (real <= pred_q90)

)


# ============================================================
# Quantile Crossing
#
# 正常：
#
# q10 <= q50 <= q90
#
# ============================================================

crossing = np.mean(

    (pred_q10 > pred_q50)

    |

    (pred_q50 > pred_q90)

)


# ============================================================
# Print Results
# ============================================================

print(
    "\n========================================"
)

print(
    "Summer 1D-CNN -> 2D-CNN"
)

print(
    "Multi-Quantile Results"
)

print(
    "========================================"
)


print(
    f"MAE (q0.5)  : {mae:.4f}"
)

print(
    f"RMSE (q0.5) : {rmse:.4f}"
)

print(
    f"R2 (q0.5)   : {r2:.4f}"
)


print(
    "\n---------- Quantile Range ----------"
)


print(
    f"Mean Range Width : "
    f"{mean_quantile_width:.4f}"
)


print(
    f"Coverage q10-q90 : "
    f"{coverage * 100:.2f}%"
)


print(
    f"Quantile Crossing: "
    f"{crossing * 100:.2f}%"
)


# ============================================================
# Results Table
# ============================================================

results = pd.DataFrame({

    'Model': [
        '1D-CNN -> 2D-CNN'
    ],

    'Season': [
        'Summer'
    ],

    'Quantiles': [
        '0.1 / 0.5 / 0.9'
    ],

    'MAE_q50': [
        mae
    ],

    'RMSE_q50': [
        rmse
    ],

    'R2_q50': [
        r2
    ],

    'Mean_Range_Width': [
        mean_quantile_width
    ],

    'Coverage_10_90': [
        coverage
    ],

    'Crossing_Rate': [
        crossing
    ]

})


print(
    "\n"
)

print(
    results
)


# ============================================================
# Plot
#
# Real
# Median q0.5
# 10%-90% Quantile Range
# ============================================================

x = np.arange(
    len(real)
)


plt.figure(
    figsize=(16, 7)
)


# -------------------------
# Quantile Range
# -------------------------

plt.fill_between(

    x,

    pred_q10,

    pred_q90,

    alpha=0.25,

    label='10%-90% Quantile Range'

)


# -------------------------
# Real
# -------------------------

plt.plot(

    x,

    real,

    label='Real',

    linewidth=1.5,

    color='black'

)


# -------------------------
# Median q0.5
# -------------------------

plt.plot(

    x,

    pred_q50,

    label='Median Prediction (q0.5)',

    linewidth=1.5,

    linestyle='--',

    color='red'

)


plt.title(

    'Summer One-hour Ahead Wind Speed Prediction\n'
    '1D-CNN -> 2D-CNN with Quantile Loss'

)


plt.xlabel(
    'Time Step'
)

plt.ylabel(
    'Wind Speed'
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()


# ============================================================
# Training / Validation Loss
# ============================================================

plt.figure(
    figsize=(10, 5)
)


plt.plot(

    history.history['loss'],

    label='Training Loss'

)


plt.plot(

    history.history['val_loss'],

    label='Validation Loss'

)


plt.title(
    'Multi-Quantile Loss'
)

plt.xlabel(
    'Epoch'
)

plt.ylabel(
    'Quantile Loss'
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# 2D-CNN -> 1D-CNN
# Summer Wind Speed Prediction
# Multi-Quantile Loss
#
# q0.1 = Lower Bound
# q0.5 = Median Prediction
# q0.9 = Upper Bound
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Conv2D,
    Conv1D,
    MaxPooling2D,
    Reshape,
    Flatten,
    Dense,
    Dropout
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# Hyper Parameters
# ============================================================

seqLength = 6

# -------------------------
# 2D-CNN
# -------------------------

filters_2d_1 = 64
filters_2d_2 = 32

kernel_size_2d_1 = (2, 2)
kernel_size_2d_2 = (2, 2)

pooling_size_2d = (2, 2)


# -------------------------
# 1D-CNN
# -------------------------

filters_1d_1 = 64
filters_1d_2 = 32

kernel_size_1d_1 = 2
kernel_size_1d_2 = 2


# -------------------------
# Training
# -------------------------

epochs = 100
batch_size = 32


# -------------------------
# Quantile
# -------------------------

quantiles = [0.1, 0.5, 0.9]


# -------------------------
# Target
# -------------------------

target_col = 'target_windspeed_1h'


# ============================================================
# Multi-Quantile Loss
# ============================================================

def multi_quantile_loss(quantiles):

    q = tf.constant(
        quantiles,
        dtype=tf.float32
    )

    def loss(y_true, y_pred):

        # y_true shape:
        # (batch, 1)
        #
        # y_pred shape:
        # (batch, 3)
        #
        # [:, 0] = q0.1
        # [:, 1] = q0.5
        # [:, 2] = q0.9

        error = y_true - y_pred

        quantile_losses = tf.maximum(
            q * error,
            (q - 1.0) * error
        )

        return tf.reduce_mean(
            quantile_losses
        )

    return loss


# ============================================================
# Sliding Window
# ============================================================

def sliding_window(X, y, seq_length):

    dataX = []
    dataY = []

    for i in range(
        0,
        len(X) - seq_length + 1
    ):

        dataX.append(
            X[i:i + seq_length]
        )

        dataY.append(
            y[i + seq_length - 1]
        )

    return (
        np.array(dataX),
        np.array(dataY).reshape(-1, 1)
    )


# ============================================================
# Build Model
#
# 2D-CNN -> 1D-CNN -> Quantile Output
# ============================================================

def build_2d_1d_cnn(
    seq_length,
    feature_count
):

    # ========================================================
    # Input
    #
    # shape:
    # (batch, seq_length, feature_count)
    # ========================================================

    inputs = Input(
        shape=(
            seq_length,
            feature_count
        )
    )


    # ========================================================
    # Reshape for Conv2D
    #
    # 原本：
    #
    # (batch, time, features)
    #
    # 變成：
    #
    # (batch, time, features, 1)
    # ========================================================

    x = Reshape(
        (
            seq_length,
            feature_count,
            1
        )
    )(inputs)


    # ========================================================
    # Part 1 : 2D-CNN
    # ========================================================

    x = Conv2D(
        filters=filters_2d_1,
        kernel_size=kernel_size_2d_1,
        activation='relu',
        padding='same'
    )(x)


    x = Conv2D(
        filters=filters_2d_2,
        kernel_size=kernel_size_2d_2,
        activation='relu',
        padding='same'
    )(x)


    x = MaxPooling2D(
        pool_size=pooling_size_2d,
        padding='same'
    )(x)


    # ========================================================
    # 取得 2D-CNN 輸出的 shape
    #
    # 例如：
    #
    # (None, 3, 8, 32)
    #
    # time      = 3
    # features  = 8
    # channels  = 32
    # ========================================================

    shape_after_2d = x.shape

    new_time = int(
        shape_after_2d[1]
    )

    new_features = int(
        shape_after_2d[2]
    )

    channels = int(
        shape_after_2d[3]
    )


    # ========================================================
    # 2D -> 1D Reshape
    #
    # (batch, time, features, channels)
    #
    # ↓
    #
    # (batch, time, features * channels)
    #
    # 例如：
    #
    # (batch, 3, 8, 32)
    #
    # ↓
    #
    # (batch, 3, 256)
    #
    # ========================================================

    x = Reshape(
        (
            new_time,
            new_features * channels
        )
    )(x)


    # ========================================================
    # Part 2 : 1D-CNN
    # ========================================================

    x = Conv1D(
        filters=filters_1d_1,
        kernel_size=kernel_size_1d_1,
        activation='relu',
        padding='same'
    )(x)


    x = Conv1D(
        filters=filters_1d_2,
        kernel_size=kernel_size_1d_2,
        activation='relu',
        padding='same'
    )(x)


    # ========================================================
    # Regression
    # ========================================================

    x = Flatten()(x)


    x = Dense(
        48,
        activation='relu'
    )(x)


    x = Dropout(
        0.2
    )(x)


    # ========================================================
    # Multi-Quantile Output
    #
    # output 0 = q0.1
    # output 1 = q0.5
    # output 2 = q0.9
    # ========================================================

    outputs = Dense(
        len(quantiles)
    )(x)


    # ========================================================
    # Model
    # ========================================================

    model = Model(
        inputs=inputs,
        outputs=outputs
    )


    # ========================================================
    # Compile
    # ========================================================

    model.compile(
        optimizer='adam',
        loss=multi_quantile_loss(
            quantiles
        )
    )


    return model


# ============================================================
# Summer Data
# ============================================================

season = 'summer'

df_train = (
    season_data[season]['df_train']
    .copy()
)

df_test = (
    season_data[season]['df_test']
    .copy()
)


# ============================================================
# X / y
# ============================================================

X_train = df_train.drop(
    columns=[
        'valid_time',
        target_col
    ]
)

y_train = df_train[
    target_col
]


X_test = df_test.drop(
    columns=[
        'valid_time',
        target_col
    ]
)

y_test = df_test[
    target_col
]


# ============================================================
# Convert Numeric
# ============================================================

X_train = X_train.apply(
    pd.to_numeric,
    errors='coerce'
)


X_test = X_test.apply(
    pd.to_numeric,
    errors='coerce'
)


y_train = pd.to_numeric(
    y_train,
    errors='coerce'
)


y_test = pd.to_numeric(
    y_test,
    errors='coerce'
)


# ============================================================
# Handle NaN
# ============================================================

X_train = (
    X_train
    .fillna(
        X_train.median()
    )
    .fillna(0)
)


X_test = (
    X_test
    .fillna(
        X_train.median()
    )
    .fillna(0)
)


y_train = y_train.fillna(
    y_train.median()
)


y_test = y_test.fillna(
    y_train.median()
)


# ============================================================
# Min-Max Normalization
#
# 只使用 Training Data 的 min / max
# 避免 Data Leakage
# ============================================================

train_min = X_train.min()

train_max = X_train.max()


denom = (
    train_max
    -
    train_min
)


denom[
    denom == 0
] = 1


trainSet = (
    (X_train - train_min)
    /
    denom
).values.astype(
    np.float32
)


testSet = (
    (X_test - train_min)
    /
    denom
).values.astype(
    np.float32
)


trainLabel = (
    y_train
    .values
    .astype(np.float32)
)


testLabel = (
    y_test
    .values
    .astype(np.float32)
)


# ============================================================
# Sliding Window
# ============================================================

trainX, trainY = sliding_window(
    trainSet,
    trainLabel,
    seqLength
)


testX, testY = sliding_window(
    testSet,
    testLabel,
    seqLength
)


# ============================================================
# Check Data Shape
# ============================================================

print(
    "\n======================================"
)

print(
    "Data Shape"
)

print(
    "======================================"
)


print(
    "trainX :",
    trainX.shape
)

print(
    "trainY :",
    trainY.shape
)


print(
    "testX  :",
    testX.shape
)

print(
    "testY  :",
    testY.shape
)


print(
    "Feature count :",
    trainX.shape[2]
)


# ============================================================
# Build Model
# ============================================================

model = build_2d_1d_cnn(
    seqLength,
    trainX.shape[2]
)


# ============================================================
# Model Summary
# ============================================================

print(
    "\n======================================"
)

print(
    "2D-CNN -> 1D-CNN"
)

print(
    "Multi-Quantile Model"
)

print(
    "======================================"
)


model.summary()


# ============================================================
# Training
# ============================================================

print(
    "\n======================================"
)

print(
    "Training Summer Model"
)

print(
    "======================================"
)


history = model.fit(

    trainX,

    trainY,

    epochs=epochs,

    batch_size=batch_size,

    validation_split=0.2,

    shuffle=False,

    verbose=1

)


# ============================================================
# Prediction
# ============================================================

prediction = model.predict(
    testX,
    verbose=0
)


# ============================================================
# Separate Quantile Predictions
# ============================================================

pred_q10 = prediction[:, 0]

pred_q50 = prediction[:, 1]

pred_q90 = prediction[:, 2]


real = testY.reshape(-1)


# ============================================================
# Point Prediction Evaluation
#
# q0.5 = Median
# ============================================================

mae = mean_absolute_error(
    real,
    pred_q50
)


rmse = np.sqrt(
    mean_squared_error(
        real,
        pred_q50
    )
)


r2 = r2_score(
    real,
    pred_q50
)


# ============================================================
# Quantile Range Width
#
# q90 - q10
# ============================================================

quantile_width = (
    pred_q90
    -
    pred_q10
)


mean_quantile_width = np.mean(
    quantile_width
)


# ============================================================
# Coverage
#
# 真實值有多少比例
# 落在 q10 ~ q90 之間
# ============================================================

coverage = np.mean(

    (real >= pred_q10)

    &

    (real <= pred_q90)

)


# ============================================================
# Quantile Crossing
#
# 正常應該：
#
# q10 <= q50 <= q90
#
# Crossing 越低越好
# ============================================================

crossing = np.mean(

    (pred_q10 > pred_q50)

    |

    (pred_q50 > pred_q90)

)


# ============================================================
# Print Results
# ============================================================

print(
    "\n========================================"
)

print(
    "Summer 2D-CNN -> 1D-CNN Results"
)

print(
    "========================================"
)


print(
    f"MAE (q0.5)      : {mae:.4f}"
)

print(
    f"RMSE (q0.5)     : {rmse:.4f}"
)

print(
    f"R2 (q0.5)       : {r2:.4f}"
)


print(
    "\n---------- Quantile Range ----------"
)


print(
    f"Mean Range Width : "
    f"{mean_quantile_width:.4f}"
)


print(
    f"Coverage q10-q90 : "
    f"{coverage * 100:.2f}%"
)


print(
    f"Quantile Crossing: "
    f"{crossing * 100:.2f}%"
)


# ============================================================
# Results Table
# ============================================================

results = pd.DataFrame({

    'Model': [
        '2D-CNN -> 1D-CNN'
    ],

    'Season': [
        'Summer'
    ],

    'Quantiles': [
        '0.1 / 0.5 / 0.9'
    ],

    'MAE_q50': [
        mae
    ],

    'RMSE_q50': [
        rmse
    ],

    'R2_q50': [
        r2
    ],

    'Mean_Range_Width': [
        mean_quantile_width
    ],

    'Coverage_10_90': [
        coverage
    ],

    'Crossing_Rate': [
        crossing
    ]

})


print(
    "\n======================================"
)

print(
    "Result Table"
)

print(
    "======================================"
)

print(
    results
)


# ============================================================
# Plot Prediction
#
# Real
# Median q0.5
# q10-q90 Quantile Range
# ============================================================

x_axis = np.arange(
    len(real)
)


plt.figure(
    figsize=(16, 7)
)


# Quantile Range
plt.fill_between(

    x_axis,

    pred_q10,

    pred_q90,

    alpha=0.25,

    label='10%-90% Quantile Range'
)


# Real
plt.plot(

    x_axis,

    real,

    label='Real',

    linewidth=1.5,

    color='black'
)


# Median
plt.plot(

    x_axis,

    pred_q50,

    label='Median Prediction (q0.5)',

    linewidth=1.5,

    linestyle='--',

    color='red'
)


plt.title(

    'Summer One-hour Ahead Wind Speed Prediction\n'
    '2D-CNN -> 1D-CNN with Quantile Loss'

)


plt.xlabel(
    'Time Step'
)


plt.ylabel(
    'Wind Speed'
)


plt.legend()


plt.grid(
    alpha=0.3
)


plt.tight_layout()


plt.show()


# ============================================================
# Plot Training / Validation Loss
# ============================================================

plt.figure(
    figsize=(10, 5)
)


plt.plot(

    history.history['loss'],

    label='Training Loss'
)


plt.plot(

    history.history['val_loss'],

    label='Validation Loss'
)


plt.title(
    '2D-CNN -> 1D-CNN Multi-Quantile Loss'
)


plt.xlabel(
    'Epoch'
)


plt.ylabel(
    'Quantile Loss'
)


plt.legend()


plt.grid(
    alpha=0.3
)


plt.tight_layout()


plt.show()

In [ ]:
# ============================================================
# NN vs DNN vs 1D-CNN
# Summer Wind Speed Prediction
#
# Compare:
# MAE / RMSE / R²
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Flatten,
    Dense,
    Dropout,
    Conv1D,
    MaxPooling1D
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# Random Seed
# 讓每次結果比較穩定
# ============================================================

np.random.seed(42)
tf.random.set_seed(42)


# ============================================================
# Hyper Parameters
# ============================================================

seqLength = 6

epochs = 100

batch_size = 32

target_col = 'target_windspeed_1h'


# ============================================================
# Sliding Window
# ============================================================

def sliding_window(X, y, seq_length):

    dataX = []
    dataY = []

    for i in range(
        len(X) - seq_length + 1
    ):

        dataX.append(
            X[i:i + seq_length]
        )

        dataY.append(
            y[i + seq_length - 1]
        )

    return (
        np.array(dataX),
        np.array(dataY).reshape(-1, 1)
    )


# ============================================================
# 1. Neural Network
#
# Input
# ↓
# Flatten
# ↓
# Dense
# ↓
# Output
# ============================================================

def build_nn(
    seq_length,
    feature_count
):

    inputs = Input(
        shape=(
            seq_length,
            feature_count
        )
    )

    x = Flatten()(inputs)

    x = Dense(
        64,
        activation='relu'
    )(x)

    outputs = Dense(1)(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name='NN'
    )

    model.compile(
        optimizer='adam',
        loss='mse'
    )

    return model


# ============================================================
# 2. Deep Neural Network
#
# Input
# ↓
# Flatten
# ↓
# Dense
# ↓
# Dense
# ↓
# Dense
# ↓
# Output
# ============================================================

def build_dnn(
    seq_length,
    feature_count
):

    inputs = Input(
        shape=(
            seq_length,
            feature_count
        )
    )

    x = Flatten()(inputs)

    x = Dense(
        128,
        activation='relu'
    )(x)

    x = Dense(
        64,
        activation='relu'
    )(x)

    x = Dense(
        32,
        activation='relu'
    )(x)

    x = Dropout(
        0.2
    )(x)

    outputs = Dense(1)(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name='DNN'
    )

    model.compile(
        optimizer='adam',
        loss='mse'
    )

    return model


# ============================================================
# 3. 1D Convolutional Neural Network
#
# Input
# ↓
# Conv1D
# ↓
# Conv1D
# ↓
# Pooling
# ↓
# Flatten
# ↓
# Dense
# ↓
# Output
# ============================================================

def build_cnn(
    seq_length,
    feature_count
):

    inputs = Input(
        shape=(
            seq_length,
            feature_count
        )
    )

    x = Conv1D(
        filters=64,
        kernel_size=2,
        activation='relu',
        padding='same'
    )(inputs)

    x = Conv1D(
        filters=32,
        kernel_size=2,
        activation='relu',
        padding='same'
    )(x)

    x = MaxPooling1D(
        pool_size=2,
        padding='same'
    )(x)

    x = Flatten()(x)

    x = Dense(
        48,
        activation='relu'
    )(x)

    x = Dropout(
        0.2
    )(x)

    outputs = Dense(1)(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name='1D-CNN'
    )

    model.compile(
        optimizer='adam',
        loss='mse'
    )

    return model


# ============================================================
# Load Summer Data
# ============================================================

season = 'summer'

df_train = (
    season_data[season]['df_train']
    .copy()
)

df_test = (
    season_data[season]['df_test']
    .copy()
)


# ============================================================
# X / y
# ============================================================

X_train = df_train.drop(
    columns=[
        'valid_time',
        target_col
    ]
)

y_train = df_train[
    target_col
]


X_test = df_test.drop(
    columns=[
        'valid_time',
        target_col
    ]
)

y_test = df_test[
    target_col
]


# ============================================================
# Convert Numeric
# ============================================================

X_train = X_train.apply(
    pd.to_numeric,
    errors='coerce'
)

X_test = X_test.apply(
    pd.to_numeric,
    errors='coerce'
)


y_train = pd.to_numeric(
    y_train,
    errors='coerce'
)

y_test = pd.to_numeric(
    y_test,
    errors='coerce'
)


# ============================================================
# Handle NaN
# ============================================================

X_train = (
    X_train
    .fillna(
        X_train.median()
    )
    .fillna(0)
)


X_test = (
    X_test
    .fillna(
        X_train.median()
    )
    .fillna(0)
)


y_train = y_train.fillna(
    y_train.median()
)


y_test = y_test.fillna(
    y_train.median()
)


# ============================================================
# Normalization
#
# IMPORTANT:
# 只使用 Training Data 的 min/max
# ============================================================

train_min = X_train.min()

train_max = X_train.max()


denom = (
    train_max
    -
    train_min
)

denom[
    denom == 0
] = 1


trainSet = (
    (X_train - train_min)
    /
    denom
).values.astype(
    np.float32
)


testSet = (
    (X_test - train_min)
    /
    denom
).values.astype(
    np.float32
)


trainLabel = (
    y_train
    .values
    .astype(np.float32)
)


testLabel = (
    y_test
    .values
    .astype(np.float32)
)


# ============================================================
# Sliding Window
# ============================================================

trainX, trainY = sliding_window(
    trainSet,
    trainLabel,
    seqLength
)


testX, testY = sliding_window(
    testSet,
    testLabel,
    seqLength
)


# ============================================================
# Shape
# ============================================================

print("\n==============================")
print("Data Shape")
print("==============================")

print(
    "trainX:",
    trainX.shape
)

print(
    "trainY:",
    trainY.shape
)

print(
    "testX :",
    testX.shape
)

print(
    "testY :",
    testY.shape
)


feature_count = trainX.shape[2]


# ============================================================
# Models
# ============================================================

models = {

    'NN':
        build_nn(
            seqLength,
            feature_count
        ),

    'DNN':
        build_dnn(
            seqLength,
            feature_count
        ),

    '1D-CNN':
        build_cnn(
            seqLength,
            feature_count
        )

}


# ============================================================
# Train + Evaluate
# ============================================================

results = []

predictions = {}

histories = {}


for model_name, model in models.items():

    print(
        "\n======================================"
    )

    print(
        f"Training: {model_name}"
    )

    print(
        "======================================"
    )

    model.summary()


    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    history = model.fit(

        trainX,

        trainY,

        epochs=epochs,

        batch_size=batch_size,

        validation_split=0.2,

        shuffle=False,

        verbose=1

    )


    histories[
        model_name
    ] = history


    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    y_pred = model.predict(
        testX,
        verbose=0
    ).reshape(-1)


    predictions[
        model_name
    ] = y_pred


    real = testY.reshape(-1)


    # --------------------------------------------------------
    # MAE
    # --------------------------------------------------------

    mae = mean_absolute_error(
        real,
        y_pred
    )


    # --------------------------------------------------------
    # RMSE
    # --------------------------------------------------------

    rmse = np.sqrt(
        mean_squared_error(
            real,
            y_pred
        )
    )


    # --------------------------------------------------------
    # R²
    # --------------------------------------------------------

    r2 = r2_score(
        real,
        y_pred
    )


    # --------------------------------------------------------
    # Save
    # --------------------------------------------------------

    results.append({

        'Model':
            model_name,

        'MAE':
            mae,

        'RMSE':
            rmse,

        'R2':
            r2

    })


# ============================================================
# Results Table
# ============================================================

results_df = pd.DataFrame(
    results
)


print(
    "\n======================================"
)

print(
    "NN vs DNN vs CNN"
)

print(
    "Summer Wind Speed Results"
)

print(
    "======================================"
)


print(
    results_df.to_string(
        index=False
    )
)


# ============================================================
# Rank
#
# R² higher = better
# ============================================================

ranking = (
    results_df
    .sort_values(
        by='R2',
        ascending=False
    )
)


print(
    "\n======================================"
)

print(
    "Ranking by R²"
)

print(
    "======================================"
)


print(
    ranking.to_string(
        index=False
    )
)


# ============================================================
# Prediction Comparison
# ============================================================

real = testY.reshape(-1)

x_axis = np.arange(
    len(real)
)


plt.figure(
    figsize=(16, 7)
)


plt.plot(
    x_axis,
    real,
    label='Real',
    linewidth=2
)


for model_name in predictions:

    plt.plot(
        x_axis,
        predictions[model_name],
        label=model_name,
        linewidth=1.2
    )


plt.title(
    'Summer Wind Speed Prediction\n'
    'NN vs DNN vs 1D-CNN'
)

plt.xlabel(
    'Time Step'
)

plt.ylabel(
    'Wind Speed'
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()


# ============================================================
# MAE Comparison
# ============================================================

plt.figure(
    figsize=(8, 5)
)

plt.bar(
    results_df['Model'],
    results_df['MAE']
)

plt.title(
    'MAE Comparison'
)

plt.ylabel(
    'MAE'
)

plt.tight_layout()

plt.show()


# ============================================================
# RMSE Comparison
# ============================================================

plt.figure(
    figsize=(8, 5)
)

plt.bar(
    results_df['Model'],
    results_df['RMSE']
)

plt.title(
    'RMSE Comparison'
)

plt.ylabel(
    'RMSE'
)

plt.tight_layout()

plt.show()


# ============================================================
# R² Comparison
# ============================================================

plt.figure(
    figsize=(8, 5)
)

plt.bar(
    results_df['Model'],
    results_df['R2']
)

plt.title(
    'R² Comparison'
)

plt.ylabel(
    'R²'
)

plt.tight_layout()

plt.show()

In [ ]:
# ============================================================
# NN vs DNN vs 1D-CNN
# Summer Wind Speed Prediction
# Multi-Quantile Loss
#
# 每個模型各自產生：
# Real
# q0.5 Median Prediction
# q0.1 ~ q0.9 Quantile Range
#
# 並比較：
# MAE / RMSE / R²
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Flatten,
    Dense,
    Dropout,
    Conv1D,
    MaxPooling1D
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# ============================================================
# Random Seed
# ============================================================

np.random.seed(42)
tf.random.set_seed(42)


# ============================================================
# Hyper Parameters
# ============================================================

seqLength = 6

epochs = 100

batch_size = 32

target_col = 'target_windspeed_1h'


# ============================================================
# Quantiles
# ============================================================

quantiles = [0.1, 0.5, 0.9]


# ============================================================
# Multi-Quantile Loss
# ============================================================

def multi_quantile_loss(quantiles):

    q = tf.constant(
        quantiles,
        dtype=tf.float32
    )

    def loss(y_true, y_pred):

        # y_true:
        # (batch, 1)
        #
        # y_pred:
        # (batch, 3)
        #
        # 0 = q0.1
        # 1 = q0.5
        # 2 = q0.9

        error = y_true - y_pred

        quantile_losses = tf.maximum(
            q * error,
            (q - 1.0) * error
        )

        return tf.reduce_mean(
            quantile_losses
        )

    return loss


# ============================================================
# Sliding Window
# ============================================================

def sliding_window(X, y, seq_length):

    dataX = []
    dataY = []

    for i in range(
        len(X) - seq_length + 1
    ):

        dataX.append(
            X[i:i + seq_length]
        )

        dataY.append(
            y[i + seq_length - 1]
        )

    return (
        np.array(dataX),
        np.array(dataY).reshape(-1, 1)
    )


# ============================================================
# Model 1 : NN
#
# Input
# ↓
# Flatten
# ↓
# Dense
# ↓
# q0.1 / q0.5 / q0.9
# ============================================================

def build_nn(
    seq_length,
    feature_count
):

    inputs = Input(
        shape=(
            seq_length,
            feature_count
        )
    )

    x = Flatten()(inputs)

    x = Dense(
        64,
        activation='relu'
    )(x)

    outputs = Dense(
        len(quantiles)
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name='NN'
    )

    model.compile(
        optimizer='adam',
        loss=multi_quantile_loss(
            quantiles
        )
    )

    return model


# ============================================================
# Model 2 : DNN
#
# Input
# ↓
# Flatten
# ↓
# Dense 128
# ↓
# Dense 64
# ↓
# Dense 32
# ↓
# q0.1 / q0.5 / q0.9
# ============================================================

def build_dnn(
    seq_length,
    feature_count
):

    inputs = Input(
        shape=(
            seq_length,
            feature_count
        )
    )

    x = Flatten()(inputs)

    x = Dense(
        128,
        activation='relu'
    )(x)

    x = Dense(
        64,
        activation='relu'
    )(x)

    x = Dense(
        32,
        activation='relu'
    )(x)

    x = Dropout(
        0.2
    )(x)

    outputs = Dense(
        len(quantiles)
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name='DNN'
    )

    model.compile(
        optimizer='adam',
        loss=multi_quantile_loss(
            quantiles
        )
    )

    return model


# ============================================================
# Model 3 : 1D-CNN
#
# Input
# ↓
# Conv1D
# ↓
# Conv1D
# ↓
# Pooling
# ↓
# Flatten
# ↓
# Dense
# ↓
# q0.1 / q0.5 / q0.9
# ============================================================

def build_cnn(
    seq_length,
    feature_count
):

    inputs = Input(
        shape=(
            seq_length,
            feature_count
        )
    )

    x = Conv1D(
        filters=64,
        kernel_size=2,
        activation='relu',
        padding='same'
    )(inputs)

    x = Conv1D(
        filters=32,
        kernel_size=2,
        activation='relu',
        padding='same'
    )(x)

    x = MaxPooling1D(
        pool_size=2,
        padding='same'
    )(x)

    x = Flatten()(x)

    x = Dense(
        48,
        activation='relu'
    )(x)

    x = Dropout(
        0.2
    )(x)

    outputs = Dense(
        len(quantiles)
    )(x)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name='1D-CNN'
    )

    model.compile(
        optimizer='adam',
        loss=multi_quantile_loss(
            quantiles
        )
    )

    return model


# ============================================================
# Load Summer Data
# ============================================================

season = 'summer'

df_train = (
    season_data[season]['df_train']
    .copy()
)

df_test = (
    season_data[season]['df_test']
    .copy()
)


# ============================================================
# X / y
# ============================================================

X_train = df_train.drop(
    columns=[
        'valid_time',
        target_col
    ]
)

y_train = df_train[
    target_col
]


X_test = df_test.drop(
    columns=[
        'valid_time',
        target_col
    ]
)

y_test = df_test[
    target_col
]


# ============================================================
# Convert Numeric
# ============================================================

X_train = X_train.apply(
    pd.to_numeric,
    errors='coerce'
)

X_test = X_test.apply(
    pd.to_numeric,
    errors='coerce'
)

y_train = pd.to_numeric(
    y_train,
    errors='coerce'
)

y_test = pd.to_numeric(
    y_test,
    errors='coerce'
)


# ============================================================
# Handle NaN
# ============================================================

X_train = (
    X_train
    .fillna(
        X_train.median()
    )
    .fillna(0)
)

X_test = (
    X_test
    .fillna(
        X_train.median()
    )
    .fillna(0)
)

y_train = y_train.fillna(
    y_train.median()
)

y_test = y_test.fillna(
    y_train.median()
)


# ============================================================
# Min-Max Normalization
#
# 只使用 Training Data
# ============================================================

train_min = X_train.min()

train_max = X_train.max()


denom = (
    train_max
    -
    train_min
)

denom[
    denom == 0
] = 1


trainSet = (
    (X_train - train_min)
    /
    denom
).values.astype(
    np.float32
)


testSet = (
    (X_test - train_min)
    /
    denom
).values.astype(
    np.float32
)


trainLabel = (
    y_train
    .values
    .astype(np.float32)
)


testLabel = (
    y_test
    .values
    .astype(np.float32)
)


# ============================================================
# Sliding Window
# ============================================================

trainX, trainY = sliding_window(
    trainSet,
    trainLabel,
    seqLength
)

testX, testY = sliding_window(
    testSet,
    testLabel,
    seqLength
)


# ============================================================
# Check Shape
# ============================================================

print("\n======================================")
print("Data Shape")
print("======================================")

print(
    "trainX:",
    trainX.shape
)

print(
    "trainY:",
    trainY.shape
)

print(
    "testX:",
    testX.shape
)

print(
    "testY:",
    testY.shape
)


feature_count = trainX.shape[2]


# ============================================================
# Models
# ============================================================

models = {

    'NN':
        build_nn(
            seqLength,
            feature_count
        ),

    'DNN':
        build_dnn(
            seqLength,
            feature_count
        ),

    '1D-CNN':
        build_cnn(
            seqLength,
            feature_count
        )

}


# ============================================================
# Containers
# ============================================================

results = []

predictions = {}

histories = {}


real = testY.reshape(-1)


# ============================================================
# Train All Models
# ============================================================

for model_name, model in models.items():

    print(
        "\n======================================"
    )

    print(
        f"Training {model_name}"
    )

    print(
        "======================================"
    )

    model.summary()


    # ========================================================
    # Train
    # ========================================================

    history = model.fit(

        trainX,

        trainY,

        epochs=epochs,

        batch_size=batch_size,

        validation_split=0.2,

        shuffle=False,

        verbose=1

    )


    histories[
        model_name
    ] = history


    # ========================================================
    # Prediction
    # ========================================================

    prediction = model.predict(
        testX,
        verbose=0
    )


    # q0.1
    pred_q10 = prediction[:, 0]

    # q0.5
    pred_q50 = prediction[:, 1]

    # q0.9
    pred_q90 = prediction[:, 2]


    predictions[
        model_name
    ] = {

        'q10':
            pred_q10,

        'q50':
            pred_q50,

        'q90':
            pred_q90

    }


    # ========================================================
    # Evaluation
    #
    # 使用 q0.5 當主要預測值
    # ========================================================

    mae = mean_absolute_error(
        real,
        pred_q50
    )


    rmse = np.sqrt(
        mean_squared_error(
            real,
            pred_q50
        )
    )


    r2 = r2_score(
        real,
        pred_q50
    )


    # ========================================================
    # Quantile Range
    # ========================================================

    range_width = (
        pred_q90
        -
        pred_q10
    )


    mean_range_width = np.mean(
        range_width
    )


    # ========================================================
    # Coverage
    # ========================================================

    coverage = np.mean(

        (real >= pred_q10)

        &

        (real <= pred_q90)

    )


    # ========================================================
    # Quantile Crossing
    # ========================================================

    crossing = np.mean(

        (pred_q10 > pred_q50)

        |

        (pred_q50 > pred_q90)

    )


    # ========================================================
    # Save Results
    # ========================================================

    results.append({

        'Model':
            model_name,

        'MAE':
            mae,

        'RMSE':
            rmse,

        'R2':
            r2,

        'Mean Range Width':
            mean_range_width,

        'Coverage':
            coverage,

        'Crossing':
            crossing

    })


# ============================================================
# Results Table
# ============================================================

results_df = pd.DataFrame(
    results
)


print(
    "\n======================================"
)

print(
    "NN vs DNN vs 1D-CNN"
)

print(
    "Summer Results"
)

print(
    "======================================"
)


print(
    results_df.to_string(
        index=False
    )
)


# ============================================================
# Ranking
# ============================================================

ranking = results_df.sort_values(
    by='R2',
    ascending=False
)


print(
    "\n======================================"
)

print(
    "Ranking by R²"
)

print(
    "======================================"
)


print(
    ranking.to_string(
        index=False
    )
)


# ============================================================
# Plot 1
#
# NN
# ============================================================

x_axis = np.arange(
    len(real)
)


plt.figure(
    figsize=(16, 7)
)


plt.fill_between(

    x_axis,

    predictions['NN']['q10'],

    predictions['NN']['q90'],

    alpha=0.25,

    label='10%-90% Quantile Range'

)


plt.plot(

    x_axis,

    real,

    label='Real',

    linewidth=1.5,

    color='black'

)


plt.plot(

    x_axis,

    predictions['NN']['q50'],

    label='Median Prediction (q0.5)',

    linewidth=1.5,

    linestyle='--',

    color='red'

)


plt.title(
    'Summer Wind Speed Prediction - NN'
)

plt.xlabel(
    'Time Step'
)

plt.ylabel(
    'Wind Speed'
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()


# ============================================================
# Plot 2
#
# DNN
# ============================================================

plt.figure(
    figsize=(16, 7)
)


plt.fill_between(

    x_axis,

    predictions['DNN']['q10'],

    predictions['DNN']['q90'],

    alpha=0.25,

    label='10%-90% Quantile Range'

)


plt.plot(

    x_axis,

    real,

    label='Real',

    linewidth=1.5,

    color='black'

)


plt.plot(

    x_axis,

    predictions['DNN']['q50'],

    label='Median Prediction (q0.5)',

    linewidth=1.5,

    linestyle='--',

    color='red'

)


plt.title(
    'Summer Wind Speed Prediction - DNN'
)

plt.xlabel(
    'Time Step'
)

plt.ylabel(
    'Wind Speed'
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()


# ============================================================
# Plot 3
#
# 1D-CNN
# ============================================================

plt.figure(
    figsize=(16, 7)
)


plt.fill_between(

    x_axis,

    predictions['1D-CNN']['q10'],

    predictions['1D-CNN']['q90'],

    alpha=0.25,

    label='10%-90% Quantile Range'

)


plt.plot(

    x_axis,

    real,

    label='Real',

    linewidth=1.5,

    color='black'

)


plt.plot(

    x_axis,

    predictions['1D-CNN']['q50'],

    label='Median Prediction (q0.5)',

    linewidth=1.5,

    linestyle='--',

    color='red'

)


plt.title(
    'Summer Wind Speed Prediction - 1D-CNN'
)

plt.xlabel(
    'Time Step'
)

plt.ylabel(
    'Wind Speed'
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.tight_layout()

plt.show()